## 프로젝트: 자율주행 보조 시스템 만들기

이 노트북은 RetinaNet 모델을 학습시키고, 학습된 모델을 사용하여 자율주행 보조 시스템을 구현 및 평가하는 전체 과정을 포함합니다.

In [5]:
# 데이터셋 처리를 위한 라이브러리를 설치해주세요.
!pip install tfrecord[torch]

### 1. 기본 라이브러리 임포트

In [1]:
# !pip install -q tfrecord pillow matplotlib torchvision --upgrade

import sys, os, torch
print("Python :", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA   :", torch.version.cuda, "| is_available:", torch.cuda.is_available())

import os, io, re, glob, time, math, itertools, copy
from typing import Dict, Any, List, Tuple

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, IterableDataset
from torchvision import transforms, models
from torchvision.ops import nms


Python : 3.12.11 | packaged by conda-forge | (main, Jun  4 2025, 14:45:31) [GCC 13.3.0]
PyTorch: 2.7.1+cu118
CUDA   : 11.8 | is_available: True


### 2. 설정 클래스 (TrainingConfig)

In [2]:
class TrainingConfig:
    ROOT_DIR = os.path.join(os.getenv("HOME") or "", "work/object_detection")

    # Data
    DATA_PATH = os.path.join(ROOT_DIR, "data")
    TRAIN_TFRECORD_PATTERN      = os.path.join(DATA_PATH, "kitti/3.2.0/kitti-train.tfrecord-*")
    VALIDATION_TFRECORD_PATTERN = os.path.join(DATA_PATH, "kitti/3.2.0/kitti-validation.tfrecord-*")
    TEST_TFRECORD_PATTERN       = os.path.join(DATA_PATH, "kitti/3.2.0/kitti-test.tfrecord-*")

    # Model
    NUM_CLASSES = 8

    # Training
    BATCH_SIZE = 2
    NUM_EPOCHS = 10
    LEARNING_RATES = [2.5e-06, 0.000625, 0.00125, 0.0025, 0.00025, 2.5e-05]
    LEARNING_RATE_BOUNDARIES = [125, 250, 500, 240000, 360000]
    OPTIMIZER_MOMENTUM = 0.9

    # Loss parameters
    CLASSIFICATION_LOSS_ALPHA = 0.25
    CLASSIFICATION_LOSS_GAMMA = 2.0
    BOX_LOSS_DELTA = 1.0

    # Decoder parameters
    CONFIDENCE_THRESHOLD = 0.05
    NMS_IOU_THRESHOLD = 0.5
    MAX_DETECTIONS_PER_CLASS = 100
    MAX_DETECTIONS = 100
    BOX_VARIANCE = [0.1, 0.1, 0.2, 0.2]

    # ✅ Checkpoint: ROOT_DIR 바로 아래에 checkpoints 폴더 생성
    CHECKPOINT_PATH = os.path.join(ROOT_DIR, 'checkpoints')
    LATEST_CHECKPOINT_FILE = os.path.join(CHECKPOINT_PATH, 'latest_checkpoint.pth')
    BEST_CHECKPOINT_FILE   = os.path.join(CHECKPOINT_PATH, 'best_checkpoint.pth')
    EPOCH_CHECKPOINT_TEMPLATE = os.path.join(CHECKPOINT_PATH, 'checkpoint_epoch_{epoch}.pth')

config = TrainingConfig()
os.makedirs(config.CHECKPOINT_PATH, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

### 3. 데이터 로딩 및 전처리

In [3]:
import hashlib

try:
    from tfrecord.torch.dataset import TFRecordDataset, MultiTFRecordDataset
except ModuleNotFoundError:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "tfrecord"])
    from tfrecord.torch.dataset import TFRecordDataset, MultiTFRecordDataset

def _index_cache_dir() -> str:
    """
    index 파일을 저장할 캐시 디렉터리.
    work/object_detection/index_cache 아래에 생성합니다.
    """
    root = os.path.join(os.getenv("HOME") or "", "work/object_detection", "index_cache")
    os.makedirs(root, exist_ok=True)
    return root

def _hashed_index_path(tfrecord_path: str) -> str:
    """
    원본 tfrecord 경로를 해시해서 index 파일명을 생성.
    파일명이 겹치지 않도록 sha1 해시를 사용합니다.
    """
    h = hashlib.sha1(tfrecord_path.encode("utf-8")).hexdigest()[:16]
    base = os.path.basename(tfrecord_path)
    return os.path.join(_index_cache_dir(), f"{base}.{h}.index")

def ensure_index_file(data_path: str) -> str:
    """
    TFRecordDataset/MultiTFRecordDataset에서 사용할 index 파일 경로 반환.
    - 이미 캐시에 존재하면 그대로 사용
    - 없으면 캐시에 새로 생성
    """
    from tfrecord.tools.tfrecord2idx import create_index

    cache_idx = _hashed_index_path(data_path)
    if os.path.exists(cache_idx):
        return cache_idx

    create_index(data_path, cache_idx)
    return cache_idx


In [4]:
class TFRecordKITTIDataset(IterableDataset):
    def __init__(self, tfrecord_pattern: str, transform=None, use_multi=False):
        super().__init__()
        self.user_transform = transform
        self.file_paths = sorted([p for p in glob.glob(tfrecord_pattern) if not p.endswith('.index')])
        if not self.file_paths:
            raise FileNotFoundError(f"No TFRecord files found for pattern: {tfrecord_pattern}")

        self.feature_description = {
            'image': "byte", 'image/file_name': "byte",
            'objects/bbox': "float", 'objects/type': "int",
        }

        # ✅ 항상 체인 방식 사용 (read-only + 캐시 인덱스와 호환)
        if use_multi:
            print("[Info] use_multi=True가 전달되었지만, read-only/캐시 인덱스 호환을 위해 "
                  "파일별 TFRecordDataset 체인 방식으로 대체합니다.")
        self.datasets = []
        for p in self.file_paths:
            idx = ensure_index_file(p)  # 캐시 디렉터리에 .index 생성/사용
            ds = TFRecordDataset(
                data_path=p,
                index_path=idx,
                description=self.feature_description,
                transform=self._parse_record
            )
            self.datasets.append(ds)

    def _parse_record(self, features: Dict[str, Any]) -> Dict[str, Any]:
        # image
        image = Image.open(io.BytesIO(features['image'])).convert('RGB')

        # filename: bytes 또는 str 모두 안전 처리
        fname_raw = features.get('image/file_name', b'')
        if isinstance(fname_raw, bytes):
            filename = fname_raw.decode('utf-8', errors='ignore')
        else:
            filename = str(fname_raw)

        # bboxes
        b_raw = features.get('objects/bbox')
        if b_raw is not None and len(b_raw) > 0:
            b = torch.tensor(b_raw, dtype=torch.float32).view(-1, 4)
            # (ymin, xmin, ymax, xmax) -> (xmin, ymin, xmax, ymax)
            b = b[:, [1, 0, 3, 2]]
        else:
            b = torch.empty((0, 4), dtype=torch.float32)

        # labels
        t_raw = features.get('objects/type')
        if t_raw is not None and len(t_raw) > 0:
            labels = torch.tensor(t_raw, dtype=torch.int64)
        else:
            labels = torch.empty((0,), dtype=torch.int64)

        sample = {'image': image, 'bbox': b, 'class_id': labels, 'filename': filename}
        if self.user_transform is not None:
            # 주의: transform은 보통 이미지(Tensor)만 변환합니다.
            sample['image'] = self.user_transform(sample['image'])
        return sample

    def __iter__(self):
        return itertools.chain.from_iterable(self.datasets)


def collate_fn(batch: List[Dict[str, Any]]) -> Tuple[torch.Tensor, List[torch.Tensor], List[torch.Tensor]]:
    images = [item['image'] for item in batch]
    bboxes = [item['bbox'] for item in batch]
    class_ids = [item['class_id'] for item in batch]

    for i, img in enumerate(images):
        if not torch.is_tensor(img) or img.dim() != 3:
            raise TypeError(f"Image at {i} is not Tensor[C,H,W]: {type(img)}, shape={getattr(img, 'shape', None)}")

    C = images[0].shape[0]
    max_h = max(int(img.shape[1]) for img in images)
    max_w = max(int(img.shape[2]) for img in images)

    padded = torch.zeros(len(images), C, max_h, max_w, dtype=images[0].dtype)
    for i, img in enumerate(images):
        _, h, w = img.shape
        padded[i, :, :h, :w] = img
    return padded, bboxes, class_ids


### 4. 유틸리티 함수 및 Anchor/Label 인코더 클래스

In [5]:
def swap_xy(boxes):  # [x,y swap]
    return torch.stack([boxes[:,1], boxes[:,0], boxes[:,3], boxes[:,2]], dim=-1)

def convert_to_xywh(boxes):
    return torch.cat([(boxes[...,:2] + boxes[...,2:]) / 2.0,
                      boxes[...,2:] - boxes[...,:2]], dim=-1)

def convert_to_corners(boxes):
    return torch.cat([boxes[...,:2] - boxes[...,2:]/2.0,
                      boxes[...,:2] + boxes[...,2:]/2.0], dim=-1)

def compute_iou(boxes1, boxes2):
    b1 = convert_to_corners(boxes1)
    b2 = convert_to_corners(boxes2)
    lu = torch.maximum(b1[:,None,:2], b2[:,:2])
    rd = torch.minimum(b1[:,None,2:], b2[:,2:])
    inter = torch.clamp(rd - lu, min=0.0)
    inter_area = inter[:,:,0] * inter[:,:,1]
    a1 = boxes1[:,2] * boxes1[:,3]
    a2 = boxes2[:,2] * boxes2[:,3]
    union = torch.clamp(a1[:,None] + a2 - inter_area, min=1e-8)
    return torch.clamp(inter_area / union, 0.0, 1.0)

class AnchorBox:
    def __init__(self):
        self.aspect_ratios = [0.5, 1.0, 2.0]
        self.scales = [2 ** x for x in [0, 1/3, 2/3]]
        self._num_anchors = len(self.aspect_ratios) * len(self.scales)
        self._strides = [2 ** i for i in range(3, 8)]
        self._areas = [x**2 for x in [32.0, 64.0, 128.0, 256.0, 512.0]]
        self._anchor_dims = self._compute_dims()

    def _compute_dims(self):
        out = []
        for area in self._areas:
            dims = []
            for r in self.aspect_ratios:
                h = torch.sqrt(torch.tensor(area / r))
                w = area / h
                for s in self.scales:
                    dims.append(torch.tensor([w*s, h*s]))
            out.append(torch.stack(dims, dim=0).view(1,1,self._num_anchors,2))
        return out

    def _get_anchors(self, fh, fw, level):
        rx = torch.arange(fw, dtype=torch.float32) + 0.5
        ry = torch.arange(fh, dtype=torch.float32) + 0.5
        centers = torch.stack(torch.meshgrid(rx, ry, indexing='xy'), dim=-1) * self._strides[level-3]
        centers = centers.unsqueeze(-2).repeat(1,1,self._num_anchors,1)
        dims = self._anchor_dims[level-3].repeat(fh, fw, 1, 1)
        anchors = torch.cat([centers, dims], dim=-1)
        return anchors.view(fh*fw*self._num_anchors, 4)

    def get_anchors(self, H, W):
        anchors = []
        for i in range(3,8):
            anchors.append(self._get_anchors(math.ceil(H/2**i), math.ceil(W/2**i), i))
        return torch.cat(anchors, dim=0)


In [6]:
class LabelEncoder:
    def __init__(self):
        self._anchor_box = AnchorBox()
        self._box_variance = torch.tensor(TrainingConfig.BOX_VARIANCE, dtype=torch.float32)

    def _match_anchor_boxes(self, anchors, gt_boxes, match_iou=0.5, ignore_iou=0.4):
        iou_matrix = compute_iou(anchors, gt_boxes)
        max_iou, matched_idx = torch.max(iou_matrix, dim=1)
        positive = max_iou >= match_iou
        negative = max_iou < ignore_iou
        ignore = ~(positive | negative)
        return matched_idx, positive.float(), ignore.float()

    def _compute_box_target(self, anchors, matched_gt):
        # xywh
        box_t = torch.cat([
            (matched_gt[:,:2] - anchors[:,:2]) / anchors[:,2:],
            torch.log(matched_gt[:,2:] / anchors[:,2:])
        ], dim=-1)
        return box_t / self._box_variance

    def _encode_single(self, image_shape, gt_boxes, cls_ids):
        anchors = self._anchor_box.get_anchors(image_shape[2], image_shape[3])
        if gt_boxes.numel() == 0:
            box_target = torch.zeros_like(anchors)
            cls_target = torch.full((anchors.shape[0],), -1.0, dtype=torch.float32)
            return box_target, cls_target

        gt_xywh = convert_to_xywh(gt_boxes)
        matched_idx, positive, ignore = self._match_anchor_boxes(anchors, gt_xywh)
        matched_gt = gt_xywh[matched_idx]
        box_target = self._compute_box_target(anchors, matched_gt)

        cls_ids = cls_ids.float()
        matched_cls = cls_ids[matched_idx]
        cls_target = torch.where(positive!=1.0, -1.0, matched_cls)
        cls_target = torch.where(ignore==1.0, -2.0, cls_target)
        return box_target, cls_target

    def encode_batch(self, images, gt_boxes_list, cls_ids_list):
        box_targets = []
        cls_targets = []
        for i in range(images.size(0)):
            b, c = self._encode_single(images.size(), gt_boxes_list[i], cls_ids_list[i])
            box_targets.append(b); cls_targets.append(c)
        box_targets = torch.stack(box_targets, dim=0)
        cls_targets = torch.stack(cls_targets, dim=0)
        return images, (cls_targets, box_targets)


### 5. RetinaNet 모델 정의

In [7]:
class ResNetBackbone(nn.Module):
    def __init__(self):
        super().__init__()
        resnet = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        self.layer1 = nn.Sequential(*list(resnet.children())[:5])  # stem + layer1
        self.layer2 = resnet.layer2
        self.layer3 = resnet.layer3
        self.layer4 = resnet.layer4

    def forward(self, x):
        c1 = self.layer1(x)
        c2 = self.layer2(c1)
        c3 = self.layer3(c2)
        c4 = self.layer4(c3)
        return c2, c3, c4

class FeaturePyramid(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone
        self.conv_c3_1x1 = nn.Conv2d( 512, 256, 1)
        self.conv_c4_1x1 = nn.Conv2d(1024, 256, 1)
        self.conv_c5_1x1 = nn.Conv2d(2048, 256, 1)
        self.conv_c3_3x3 = nn.Conv2d(256, 256, 3, padding=1)
        self.conv_c4_3x3 = nn.Conv2d(256, 256, 3, padding=1)
        self.conv_c5_3x3 = nn.Conv2d(256, 256, 3, padding=1)
        self.conv_c6_3x3 = nn.Conv2d(2048, 256, 3, stride=2, padding=1)
        self.conv_c7_3x3 = nn.Conv2d(256, 256, 3, stride=2, padding=1)

    def forward(self, x):
        c3, c4, c5 = self.backbone(x)
        p5 = self.conv_c5_1x1(c5)
        p4_lat = self.conv_c4_1x1(c4)
        p4  = p4_lat + F.interpolate(p5, size=p4_lat.shape[2:], mode='nearest')
        p3_lat = self.conv_c3_1x1(c3)
        p3  = p3_lat + F.interpolate(p4, size=p3_lat.shape[2:], mode='nearest')
        p3 = self.conv_c3_3x3(p3); p4 = self.conv_c4_3x3(p4); p5 = self.conv_c5_3x3(p5)
        p6 = self.conv_c6_3x3(c5)
        p7 = self.conv_c7_3x3(F.relu(p6))
        return p3, p4, p5, p6, p7

def build_head(out_filters, bias_init):
    layers = []
    for _ in range(4):
        layers += [nn.Conv2d(256,256,3,padding=1), nn.ReLU()]
    last = nn.Conv2d(256, out_filters, 3, padding=1)
    if isinstance(bias_init, float):
        nn.init.constant_(last.bias, bias_init)
    else:
        nn.init.constant_(last.bias, 0.0)
    layers.append(last)
    return nn.Sequential(*layers)

class RetinaNet(nn.Module):
    def __init__(self, num_classes, backbone=None):
        super().__init__()
        self.num_classes = num_classes
        self.fpn = FeaturePyramid(backbone or ResNetBackbone())
        prior_probability = -torch.log(torch.tensor((1 - 0.01) / 0.01))
        self.cls_head = build_head(9*num_classes, float(prior_probability))
        self.box_head = build_head(9*4, "zeros")

    def forward(self, x):
        feats = self.fpn(x)
        N = x.size(0)
        cls_outs, box_outs = [], []
        for f in feats:
            box_outs.append(self.box_head(f).view(N, -1, 4))
            cls_outs.append(self.cls_head(f).view(N, -1, self.num_classes))
        return torch.cat(cls_outs, 1), torch.cat(box_outs, 1)


## 디코더 & 시각화

In [8]:
class DecodePredictions(nn.Module):
    def __init__(self):
        super().__init__()
        self.num_classes = TrainingConfig.NUM_CLASSES
        self.confidence_threshold = TrainingConfig.CONFIDENCE_THRESHOLD
        self.nms_iou_threshold = TrainingConfig.NMS_IOU_THRESHOLD
        self.max_detections_per_class = TrainingConfig.MAX_DETECTIONS_PER_CLASS
        self.max_detections = TrainingConfig.MAX_DETECTIONS
        self._anchors = AnchorBox()
        self._box_variance = torch.tensor(TrainingConfig.BOX_VARIANCE, dtype=torch.float32)

    def _decode(self, anchors, box_pred):
        boxes = box_pred * self._box_variance
        boxes = torch.cat([
            boxes[:, :, :2] * anchors[:, :, 2:] + anchors[:, :, :2],
            torch.exp(boxes[:, :, 2:]) * anchors[:, :, 2:]
        ], dim=-1)
        return convert_to_corners(boxes)

    def forward(self, images, preds):
        cls_logits, box_preds = preds
        cls_scores = torch.sigmoid(cls_logits)
        H, W = images.shape[2], images.shape[3]
        anchors = self._anchors.get_anchors(H, W)[None, ...]  # [1, A, 4]
        boxes = self._decode(anchors, box_preds)  # [B, A, 4]

        out_boxes, out_scores, out_classes = [], [], []
        for b in range(images.size(0)):
            bboxes = boxes[b]
            scores = cls_scores[b]  # [A, C]
            for c in range(self.num_classes):
                cs = scores[:, c]
                mask = cs > self.confidence_threshold
                if not mask.any():
                    continue
                c_boxes = bboxes[mask]; c_scores = cs[mask]
                if c_scores.numel() > self.max_detections_per_class:
                    topk = torch.topk(c_scores, self.max_detections_per_class)
                    c_scores = topk.values; c_boxes = c_boxes[topk.indices]
                keep = nms(c_boxes, c_scores, self.nms_iou_threshold)
                out_boxes.append(c_boxes[keep])
                out_scores.append(c_scores[keep])
                out_classes.append(torch.full_like(c_scores[keep], c, dtype=torch.int64))
        if not out_boxes:
            return torch.empty(0,4), torch.empty(0), torch.empty(0, dtype=torch.int64)
        fb = torch.cat(out_boxes,0); fs = torch.cat(out_scores,0); fc = torch.cat(out_classes,0)
        if fs.numel() > self.max_detections:
            topk = torch.topk(fs, self.max_detections)
            fb, fs, fc = fb[topk.indices], topk.values, fc[topk.indices]
        return fb, fs, fc

def visualize_detections(image, boxes, classes, scores, figsize=(7,7), linewidth=1):
    img = np.array(image, dtype=np.uint8)
    plt.figure(figsize=figsize); plt.axis("off"); plt.imshow(img)
    ax = plt.gca()
    for box, c, s in zip(boxes, classes, scores):
        x1,y1,x2,y2 = box
        w,h = x2-x1, y2-y1
        rect = plt.Rectangle((x1,y1), w,h, fill=False, linewidth=linewidth)
        ax.add_patch(rect)
        ax.text(x1, y1, f"{int(c)}:{s:.2f}", bbox={"facecolor":"white","alpha":0.5}, fontsize=8)
    plt.show()


### 6. 손실 함수 (Loss Function)

In [9]:
class RetinaNetBoxLoss(nn.Module):
    def __init__(self, delta):
        super().__init__()
        self.delta = delta
    def forward(self, y_true, y_pred):
        d = y_true - y_pred
        ad = torch.abs(d)
        loss = torch.where(ad < self.delta, 0.5 * (d**2), ad - 0.5)
        return torch.sum(loss, dim=-1)

class RetinaNetClassificationLoss(nn.Module):
    def __init__(self, alpha, gamma):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
    def forward(self, y_true, y_pred):
        ce = F.binary_cross_entropy_with_logits(y_pred, y_true, reduction='none')
        p  = torch.sigmoid(y_pred)
        alpha = torch.where(y_true == 1.0, self.alpha, 1.0 - self.alpha)
        pt = torch.where(y_true == 1.0, p, 1.0 - p)
        loss = alpha * (1.0 - pt) ** self.gamma * ce
        return torch.sum(loss, dim=-1)

class RetinaNetLoss(nn.Module):
    def __init__(self, num_classes=TrainingConfig.NUM_CLASSES):
        super().__init__()
        self.num_classes = num_classes
        self.clf = RetinaNetClassificationLoss(TrainingConfig.CLASSIFICATION_LOSS_ALPHA,
                                               TrainingConfig.CLASSIFICATION_LOSS_GAMMA)
        self.box = RetinaNetBoxLoss(TrainingConfig.BOX_LOSS_DELTA)
    def forward(self, preds, targets):
        cls_logits, box_preds = preds
        cls_targets, box_targets = targets
        positive_mask = (cls_targets > -1.0).float()
        ignore_mask   = (cls_targets == -2.0).float()
        normalizer = torch.sum(positive_mask, dim=-1).clamp(min=1.0)

        temp = torch.where(cls_targets >= 0, cls_targets.long(), 0)
        one_hot = F.one_hot(temp, num_classes=self.num_classes).float()
        one_hot = one_hot * positive_mask.unsqueeze(-1)

        clf_loss = self.clf(one_hot, cls_logits.float())
        clf_loss = torch.where(ignore_mask==1.0, 0.0, clf_loss)
        clf_loss = torch.sum(clf_loss, dim=-1) / normalizer

        box_loss = self.box(box_targets.float(), box_preds.float())
        box_loss = torch.where(positive_mask==1.0, box_loss, 0.0)
        box_loss = torch.sum(box_loss, dim=-1) / normalizer

        return clf_loss + box_loss


In [10]:
# ⬇️ 기존 pil_to_tensor 앞/뒤 아무데나 추가
from torchvision import transforms

IMG_SIZE = 512  # 384~512 권장

pil_to_tensor = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),   # ✅ 고정 리사이즈로 피처맵/앵커 수 감소
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406],
                         std =[0.229,0.224,0.225]),
])

train_ds = TFRecordKITTIDataset(config.TRAIN_TFRECORD_PATTERN, transform=pil_to_tensor, use_multi=True)
val_ds   = TFRecordKITTIDataset(config.VALIDATION_TFRECORD_PATTERN, transform=pil_to_tensor, use_multi=True)

train_loader = DataLoader(train_ds, batch_size=config.BATCH_SIZE, collate_fn=collate_fn, num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=config.BATCH_SIZE, collate_fn=collate_fn, num_workers=0)

model = RetinaNet(config.NUM_CLASSES).to(device)
label_encoder = LabelEncoder()
loss_fn = RetinaNetLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=config.LEARNING_RATES[0], momentum=config.OPTIMIZER_MOMENTUM)

def lr_lambda(epoch):
    for i,b in enumerate(config.LEARNING_RATE_BOUNDARIES):
        if epoch < b:
            return config.LEARNING_RATES[i] / config.LEARNING_RATES[0]
    return config.LEARNING_RATES[-1] / config.LEARNING_RATES[0]

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


[Info] use_multi=True가 전달되었지만, read-only/캐시 인덱스 호환을 위해 파일별 TFRecordDataset 체인 방식으로 대체합니다.
[Info] use_multi=True가 전달되었지만, read-only/캐시 인덱스 호환을 위해 파일별 TFRecordDataset 체인 방식으로 대체합니다.


In [12]:
from torch.cuda.amp import autocast, GradScaler

scaler = GradScaler(enabled=torch.cuda.is_available())

start_epoch, best_val = 0, float('inf')

if os.path.exists(config.LATEST_CHECKPOINT_FILE):
    ckpt = torch.load(config.LATEST_CHECKPOINT_FILE, map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    start_epoch = ckpt['epoch'] + 1
    print(f"[Resume] start from epoch {start_epoch}")

for epoch in range(start_epoch, config.NUM_EPOCHS):
    t0 = time.time()
    model.train()
    tr_loss = 0.0

    for i, (imgs, gt_boxes, cls_ids) in enumerate(train_loader):
        imgs = imgs.to(device, non_blocking=True)

        # encode는 CPU 텐서 목록이므로, 결과만 GPU로 보냅니다.
        imgs, (cls_t, box_t) = label_encoder.encode_batch(imgs, gt_boxes, cls_ids)
        cls_t, box_t = cls_t.to(device, non_blocking=True), box_t.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        # ✅ AMP
        with autocast(enabled=torch.cuda.is_available()):
            pred = model(imgs)
            loss = torch.mean(loss_fn(pred, (cls_t, box_t)))

        # ✅ AMP 스케일링
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        tr_loss += float(loss.detach().item())

        if (i+1) % 10 == 0:
            print(f"Epoch {epoch+1}/{config.NUM_EPOCHS} | Step {i+1} | loss {loss.item():.4f}")

        # 메모리 단편화 완화(선택)
        del imgs, cls_t, box_t, pred, loss
        torch.cuda.empty_cache()

    tr_loss /= max(1, len(train_loader))

    # ===== Validation =====
    model.eval()
    va_loss = 0.0
    with torch.no_grad():
        for imgs, gt_boxes, cls_ids in val_loader:
            imgs = imgs.to(device, non_blocking=True)
            imgs, (cls_t, box_t) = label_encoder.encode_batch(imgs, gt_boxes, cls_ids)
            cls_t, box_t = cls_t.to(device, non_blocking=True), box_t.to(device, non_blocking=True)
            with autocast(enabled=torch.cuda.is_available()):
                pred = model(imgs)
                loss = torch.mean(loss_fn(pred, (cls_t, box_t)))
            va_loss += float(loss.detach().item())

            del imgs, cls_t, box_t, pred, loss
            torch.cuda.empty_cache()

    va_loss /= max(1, len(val_loader))
    dt = time.time() - t0
    print(f"[Epoch {epoch+1}] train {tr_loss:.4f} | val {va_loss:.4f} | {dt:.1f}s")

    scheduler.step()

    # latest
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict()
    }, config.LATEST_CHECKPOINT_FILE)

    # per-epoch
    torch.save({'epoch': epoch, 'model_state_dict': model.state_dict()},
               config.EPOCH_CHECKPOINT_TEMPLATE.format(epoch=epoch+1))

    if va_loss < best_val:
        best_val = va_loss
        torch.save({'epoch': epoch, 'model_state_dict': model.state_dict()}, config.BEST_CHECKPOINT_FILE)
        print(f"** New Best Saved (val {va_loss:.4f}) **")


/tmp/ipykernel_1063/4022393619.py:3: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=torch.cuda.is_available())
/tmp/ipykernel_1063/4022393619.py:29: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=torch.cuda.is_available()):


Epoch 1/10 | Step 10 | loss 0.2964
Epoch 1/10 | Step 20 | loss 0.2964
Epoch 1/10 | Step 30 | loss 0.2965
Epoch 1/10 | Step 40 | loss 0.2964
Epoch 1/10 | Step 50 | loss 0.2964
Epoch 1/10 | Step 60 | loss 0.2964
Epoch 1/10 | Step 70 | loss 0.2964
Epoch 1/10 | Step 80 | loss 0.2964
Epoch 1/10 | Step 90 | loss 0.2964
Epoch 1/10 | Step 100 | loss 0.2964
Epoch 1/10 | Step 110 | loss 0.2965
Epoch 1/10 | Step 120 | loss 0.2964
Epoch 1/10 | Step 130 | loss 0.2964
Epoch 1/10 | Step 140 | loss 0.2964
Epoch 1/10 | Step 150 | loss 0.2964
Epoch 1/10 | Step 160 | loss 0.2964
Epoch 1/10 | Step 170 | loss 0.2965
Epoch 1/10 | Step 180 | loss 0.2964
Epoch 1/10 | Step 190 | loss 0.2964
Epoch 1/10 | Step 200 | loss 0.2964
Epoch 1/10 | Step 210 | loss 0.2964
Epoch 1/10 | Step 220 | loss 0.2964
Epoch 1/10 | Step 230 | loss 0.2964
Epoch 1/10 | Step 240 | loss 0.2964
Epoch 1/10 | Step 250 | loss 0.2964
Epoch 1/10 | Step 260 | loss 0.2963
Epoch 1/10 | Step 270 | loss 0.2964
Epoch 1/10 | Step 280 | loss 0.2964
E

KeyboardInterrupt: 

In [ ]:
test_ds = TFRecordKITTIDataset(config.TEST_TFRECORD_PATTERN, transform=pil_to_tensor, use_multi=True)
test_loader = DataLoader(test_ds, batch_size=config.BATCH_SIZE, collate_fn=collate_fn, num_workers=0)
print("Test size:", sum(1 for _ in test_loader))

ckpt_paths = sorted(glob.glob(config.EPOCH_CHECKPOINT_TEMPLATE.format(epoch='*')),
                    key=lambda x: int(re.search(r'checkpoint_epoch_(\d+).pth', x).group(1)))

if not ckpt_paths:
    print("No epoch checkpoints found.")
else:
    for p in ckpt_paths:
        ckpt = torch.load(p, map_location=device)
        model.load_state_dict(ckpt['model_state_dict'])
        model.eval()
        total = 0.0
        with torch.no_grad():
            for imgs, gt_boxes, cls_ids in test_loader:
                imgs = imgs.to(device)
                imgs, (cls_t, box_t) = label_encoder.encode_batch(imgs, gt_boxes, cls_ids)
                cls_t, box_t = cls_t.to(device), box_t.to(device)
                pred = model(imgs)
                loss = torch.mean(loss_fn(pred, (cls_t, box_t)))
                total += loss.item()
        print(f"{os.path.basename(p)} | test loss {total / max(1,len(test_loader)):.4f}")


In [ ]:
# decode + visualize one batch
decoder = DecodePredictions()

def show_one_batch():
    model.eval()
    imgs, gt_boxes, cls_ids = next(iter(val_loader))
    with torch.no_grad():
        imgs_dev = imgs.to(device)
        preds = model(imgs_dev)
        boxes, scores, classes = decoder(imgs_dev, preds)

    # 첫 이미지 기준으로 필터링 (여기서는 NMS 후 전체 병합이라 예시로 상위 20개 표시)
    topk = min(20, scores.numel())
    if topk == 0:
        print("No detections over threshold.")
        return
    t = torch.topk(scores, topk)
    bxs = boxes[t.indices].cpu().numpy()
    scs = scores[t.indices].cpu().numpy()
    cls = classes[t.indices].cpu().numpy()

    # 정규화를 역변환해서 시각화(간단히 값 범위만 클램프)
    inv = imgs[0].permute(1,2,0).cpu().numpy()
    inv = (inv * np.array([0.229,0.224,0.225]) + np.array([0.485,0.456,0.406]))
    inv = (np.clip(inv, 0, 1)*255).astype(np.uint8)

    visualize_detections(inv, bxs, cls, scs, figsize=(7,7), linewidth=2)

show_one_batch()


In [ ]:
BATCH_SIZE = 2
loader = DataLoader(val_ds, batch_size=BATCH_SIZE, collate_fn=collate_fn)
try:
    real_images, real_gt_boxes, real_cls_ids = next(iter(loader))
except StopIteration:
    raise RuntimeError("Validation TFRecord가 비어 있습니다.")

_, (cls_targets, box_targets) = label_encoder.encode_batch(real_images, real_gt_boxes, real_cls_ids)
cls_targets = cls_targets.to(device); box_targets = box_targets.to(device)

num_anchors = cls_targets.shape[1]
pred_cls = torch.randn(BATCH_SIZE, num_anchors, config.NUM_CLASSES, device=device)
pred_box = torch.randn(BATCH_SIZE, num_anchors, 4, device=device)

clf_loss_fn = RetinaNetClassificationLoss(config.CLASSIFICATION_LOSS_ALPHA, config.CLASSIFICATION_LOSS_GAMMA)
box_loss_fn = RetinaNetBoxLoss(config.BOX_LOSS_DELTA)
total_loss_fn = RetinaNetLoss()

positive_mask = (cls_targets > -1.0).float()
ignore_mask   = (cls_targets == -2.0).float()
normalizer = torch.sum(positive_mask, dim=-1).clamp(min=1.0)

temp = torch.where(cls_targets >= 0, cls_targets.long(), 0)
one_hot = F.one_hot(temp, num_classes=config.NUM_CLASSES).float() * positive_mask.unsqueeze(-1)

clf_loss = clf_loss_fn(one_hot, pred_cls.float())
clf_loss = torch.where(ignore_mask==1.0, 0.0, clf_loss)
clf_loss_per_sample = torch.sum(clf_loss, dim=-1) / normalizer

box_loss = box_loss_fn(box_targets.float(), pred_box.float())
box_loss = torch.where(positive_mask==1.0, box_loss, 0.0)
box_loss_per_sample = torch.sum(box_loss, dim=-1) / normalizer

total_loss = total_loss_fn((pred_cls, pred_box), (cls_targets, box_targets))

print("Classification per-sample:", clf_loss_per_sample.detach().cpu().numpy())
print("Box per-sample          :", box_loss_per_sample.detach().cpu().numpy())
print("Combined (manual)       :", (clf_loss_per_sample+box_loss_per_sample).detach().cpu().numpy())
print("Total (class)           :", total_loss.detach().cpu().numpy())


### 9. 프로젝트: 자율주행 보조 시스템 구현 및 평가

In [ ]:
print('
--- 추론 및 평가 단계 ---')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if not os.path.exists(config.BEST_CHECKPOINT_FILE):
    print('오류:', config.BEST_CHECKPOINT_FILE, '에서 모델 체크포인트를 찾을 수 없습니다.')
    print('먼저 위 셀에서 모델 학습을 완료해주세요.')
else:
    print('Best 체크포인트 파일을 로드합니다:', config.BEST_CHECKPOINT_FILE)
    inference_model = RetinaNet(config.NUM_CLASSES, get_backbone()).to(device)
    checkpoint = torch.load(config.BEST_CHECKPOINT_FILE, map_location=device)
    inference_model.load_state_dict(checkpoint['model_state_dict'])
    inference_model.eval()
    decoder = DecodePredictions()
    print('추론 모델이 성공적으로 준비되었습니다.')

def autodrive_system(image_pil):
    image_tensor = transforms.ToTensor()(image_pil).unsqueeze(0).to(device)
    with torch.no_grad():
        boxes, scores, classes = decoder(image_tensor, inference_model(image_tensor))
    boxes, classes = boxes.cpu().numpy(), classes.cpu().numpy()
    if 3 in classes: # 'Pedestrian'
        print('보행자가 감지되었습니다.')
        return 'Stop'
    for box, cls in zip(boxes, classes):
        if cls in [0, 1, 2]: # 'Car', 'Van', 'Truck'
            if (box[2] - box[0]) >= 300 or (box[3] - box[1]) >= 300:
                print('큰 차량(class:', cls, ')이 감지되었습니다.')
                return 'Stop'
    return 'Go'

def test_system():
    test_cases = {f'data/go_{i}.png': 'Go' for i in range(1, 6)}
    test_cases.update({f'data/stop_{i}.png': 'Stop' for i in range(1, 6)})
    score = 0
    print('
--- 시스템 평가 시작 ---')
    for image_path, expected in test_cases.items():
        full_path = os.path.join(config.ROOT_DIR, image_path)
        try:
            result = autodrive_system(Image.open(full_path))
            print('테스트:', full_path, '| 기대값=' + expected, '| 결과=' + result, end='')
            if result == expected:
                score += 10
                print(' -> 정답!')
            else:
                print(' -> 오답!')
        except Exception as e:
            print('오류 (', full_path, '):', e)
    print(f'
최종 점수: {score} / 100')
    print('--- 평가 종료 ---')

if 'inference_model' in locals():
    test_system()
else:
    print('추론 모델이 준비되지 않아 평가를 실행할 수 없습니다.')

In [3]:
# !pip install -q tfrecord pillow matplotlib torchvision --upgrade

import sys, os, torch
print("Python :", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA   :", torch.version.cuda, "| is_available:", torch.cuda.is_available())

import os, io, re, glob, time, math, itertools, copy
from typing import Dict, Any, List, Tuple

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, IterableDataset
from torchvision import transforms, models
from torchvision.ops import nms

# =========================
# Config
# =========================
class TrainingConfig:
    ROOT_DIR = os.path.join(os.getenv("HOME") or "", "work/object_detection")

    # Data
    DATA_PATH = os.path.join(ROOT_DIR, "data")
    TRAIN_TFRECORD_PATTERN      = os.path.join(DATA_PATH, "kitti/3.2.0/kitti-train.tfrecord-*")
    VALIDATION_TFRECORD_PATTERN = os.path.join(DATA_PATH, "kitti/3.2.0/kitti-validation.tfrecord-*")
    TEST_TFRECORD_PATTERN       = os.path.join(DATA_PATH, "kitti/3.2.0/kitti-test.tfrecord-*")

    # Model
    NUM_CLASSES = 8

    # Training
    BATCH_SIZE = 16
    NUM_EPOCHS = 10
    LEARNING_RATES = [2.5e-06, 0.000625, 0.00125, 0.0025, 0.00025, 2.5e-05]
    LEARNING_RATE_BOUNDARIES = [125, 250, 500, 240000, 360000]  # (주의: epoch 기준)
    OPTIMIZER_MOMENTUM = 0.9

    # Loss parameters
    CLASSIFICATION_LOSS_ALPHA = 0.25
    CLASSIFICATION_LOSS_GAMMA = 2.0
    BOX_LOSS_DELTA = 1.0

    # Decoder parameters
    CONFIDENCE_THRESHOLD = 0.05
    NMS_IOU_THRESHOLD = 0.5
    MAX_DETECTIONS_PER_CLASS = 100
    MAX_DETECTIONS = 100
    BOX_VARIANCE = [0.1, 0.1, 0.2, 0.2]

    # Checkpoint
    CHECKPOINT_PATH = os.path.join(ROOT_DIR, 'checkpoints')
    LATEST_CHECKPOINT_FILE = os.path.join(CHECKPOINT_PATH, 'latest_checkpoint.pth')
    BEST_CHECKPOINT_FILE   = os.path.join(CHECKPOINT_PATH, 'best_checkpoint.pth')
    EPOCH_CHECKPOINT_TEMPLATE = os.path.join(CHECKPOINT_PATH, 'checkpoint_epoch_{epoch}.pth')

config = TrainingConfig()
os.makedirs(config.CHECKPOINT_PATH, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

# =========================
# TFRecord helpers
# =========================
import hashlib

try:
    from tfrecord.torch.dataset import TFRecordDataset, MultiTFRecordDataset
except ModuleNotFoundError:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "tfrecord"])
    from tfrecord.torch.dataset import TFRecordDataset, MultiTFRecordDataset

def _index_cache_dir() -> str:
    """
    index 파일을 저장할 캐시 디렉터리: ~/work/object_detection/index_cache
    """
    root = os.path.join(os.getenv("HOME") or "", "work/object_detection", "index_cache")
    os.makedirs(root, exist_ok=True)
    return root

def _hashed_index_path(tfrecord_path: str) -> str:
    """
    원본 tfrecord 경로를 해시해서 index 파일명을 생성.
    """
    h = hashlib.sha1(tfrecord_path.encode("utf-8")).hexdigest()[:16]
    base = os.path.basename(tfrecord_path)
    return os.path.join(_index_cache_dir(), f"{base}.{h}.index")

import numpy as np

def ensure_index_file_or_none(data_path: str) -> str | None:
    """
    캐시 .index가 유효하면 경로 반환.
    - 비어있거나 파싱 불가하면 재생성.
    - 재생성 후에도 유효하지 않으면 None 반환(→ 인덱스 없이 순차 읽기).
    """
    cache_idx = _hashed_index_path(data_path)

    def _valid(idx_path: str) -> bool:
        try:
            if not (os.path.exists(idx_path) and os.path.getsize(idx_path) > 0):
                return False
            arr = np.loadtxt(idx_path, dtype=np.int64)
            # reader.py가 [:,0] 인덱싱을 하므로 최소 2D/열>=1 필요
            return (arr.ndim == 2) and (arr.shape[1] >= 1) and (arr.size > 0)
        except Exception:
            return False

    if not _valid(cache_idx):
        try:
            from tfrecord.tools.tfrecord2idx import create_index
            create_index(data_path, cache_idx)
        except Exception as e:
            print(f"[Warn] index create failed for {data_path}: {e}. Use sequential read.")
            return None
        if not _valid(cache_idx):
            print(f"[Warn] invalid/empty index at {cache_idx}. Use sequential read.")
            return None

    return cache_idx


# =========================
# Dataset / Collate
# =========================
class TFRecordKITTIDataset(IterableDataset):
    def __init__(self, tfrecord_pattern: str, transform=None, use_multi=False):
        super().__init__()
        self.user_transform = transform
        self.file_paths = sorted([p for p in glob.glob(tfrecord_pattern) if not p.endswith('.index')])
        if not self.file_paths:
            raise FileNotFoundError(f"No TFRecord files found for pattern: {tfrecord_pattern}")

        self.feature_description = {
            'image': "byte", 'image/file_name': "byte",
            'objects/bbox': "float", 'objects/type': "int",
        }

        # 항상 체인 방식 사용 (read-only + 캐시 인덱스와 호환)
        if use_multi:
            print("[Info] use_multi=True가 전달되었지만, read-only/캐시 인덱스 호환을 위해 "
                  "파일별 TFRecordDataset 체인 방식으로 대체합니다.")
        self.datasets = []
        for p in self.file_paths:
            idx = ensure_index_file_or_none(p)  # ✅ 유효하지 않으면 None
            kwargs = dict(
                data_path=p,
                description=self.feature_description,
                transform=self._parse_record,
            )
            if idx is not None:
                kwargs["index_path"] = idx
                kwargs["shuffle_queue_size"] = 1024  # 인덱스 있을 때만 큐 셔플
            else:
                kwargs["index_path"] = None          # ✅ 순차 읽기(인덱스 미사용)
                kwargs["shuffle_queue_size"] = 0     # 인덱스 없을 땐 셔플 버퍼 끔
        
            ds = TFRecordDataset(**kwargs)
            self.datasets.append(ds)


    def _parse_record(self, features: Dict[str, Any]) -> Dict[str, Any]:
        # image
        image = Image.open(io.BytesIO(features['image'])).convert('RGB')

        # filename: bytes 또는 str 모두 안전 처리
        fname_raw = features.get('image/file_name', b'')
        if isinstance(fname_raw, bytes):
            filename = fname_raw.decode('utf-8', errors='ignore')
        else:
            filename = str(fname_raw)

        # bboxes
        b_raw = features.get('objects/bbox')
        if b_raw is not None and len(b_raw) > 0:
            b = torch.tensor(b_raw, dtype=torch.float32).view(-1, 4)
            # (ymin, xmin, ymax, xmax) -> (xmin, ymin, xmax, ymax)
            b = b[:, [1, 0, 3, 2]]
        else:
            b = torch.empty((0, 4), dtype=torch.float32)

        # labels
        t_raw = features.get('objects/type')
        if t_raw is not None and len(t_raw) > 0:
            labels = torch.tensor(t_raw, dtype=torch.int64)
        else:
            labels = torch.empty((0,), dtype=torch.int64)

        sample = {'image': image, 'bbox': b, 'class_id': labels, 'filename': filename}
        if self.user_transform is not None:
            # transform은 보통 이미지(Tensor)만 변환
            sample['image'] = self.user_transform(sample['image'])
        return sample

    def __iter__(self):
        return itertools.chain.from_iterable(self.datasets)

def collate_fn(batch: List[Dict[str, Any]]) -> Tuple[torch.Tensor, List[torch.Tensor], List[torch.Tensor]]:
    images = [item['image'] for item in batch]
    bboxes = [item['bbox'] for item in batch]
    class_ids = [item['class_id'] for item in batch]

    for i, img in enumerate(images):
        if not torch.is_tensor(img) or img.dim() != 3:
            raise TypeError(f"Image at {i} is not Tensor[C,H,W]: {type(img)}, shape={getattr(img, 'shape', None)}")

    C = images[0].shape[0]
    max_h = max(int(img.shape[1]) for img in images)
    max_w = max(int(img.shape[2]) for img in images)

    padded = torch.zeros(len(images), C, max_h, max_w, dtype=images[0].dtype)
    for i, img in enumerate(images):
        _, h, w = img.shape
        padded[i, :, :h, :w] = img
    return padded, bboxes, class_ids

# =========================
# Geometry / Anchors
# =========================
def swap_xy(boxes):  # [x,y swap]
    return torch.stack([boxes[:,1], boxes[:,0], boxes[:,3], boxes[:,2]], dim=-1)

def convert_to_xywh(boxes):
    return torch.cat([(boxes[...,:2] + boxes[...,2:]) / 2.0,
                      boxes[...,2:] - boxes[...,:2]], dim=-1)

def convert_to_corners(boxes):
    return torch.cat([boxes[...,:2] - boxes[...,2:]/2.0,
                      boxes[...,:2] + boxes[...,2:]/2.0], dim=-1)

def compute_iou(boxes1, boxes2):
    b1 = convert_to_corners(boxes1)
    b2 = convert_to_corners(boxes2)
    lu = torch.maximum(b1[:,None,:2], b2[:,:2])
    rd = torch.minimum(b1[:,None,2:], b2[:,2:])
    inter = torch.clamp(rd - lu, min=0.0)
    inter_area = inter[:,:,0] * inter[:,:,1]
    a1 = boxes1[:,2] * boxes1[:,3]
    a2 = boxes2[:,2] * boxes2[:,3]
    union = torch.clamp(a1[:,None] + a2 - inter_area, min=1e-8)
    return torch.clamp(inter_area / union, 0.0, 1.0)

class AnchorBox:
    def __init__(self):
        self.aspect_ratios = [0.5, 1.0, 2.0]
        self.scales = [2 ** x for x in [0, 1/3, 2/3]]
        self._num_anchors = len(self.aspect_ratios) * len(self.scales)
        self._strides = [2 ** i for i in range(3, 8)]
        self._areas = [x**2 for x in [32.0, 64.0, 128.0, 256.0, 512.0]]
        self._anchor_dims = self._compute_dims()

    def _compute_dims(self):
        out = []
        for area in self._areas:
            dims = []
            for r in self.aspect_ratios:
                h = torch.sqrt(torch.tensor(area / r))
                w = area / h
                for s in self.scales:
                    dims.append(torch.tensor([w*s, h*s]))
            out.append(torch.stack(dims, dim=0).view(1,1,self._num_anchors,2))
        return out

    def _get_anchors(self, fh, fw, level):
        rx = torch.arange(fw, dtype=torch.float32) + 0.5
        ry = torch.arange(fh, dtype=torch.float32) + 0.5
        centers = torch.stack(torch.meshgrid(rx, ry, indexing='xy'), dim=-1) * self._strides[level-3]
        centers = centers.unsqueeze(-2).repeat(1,1,self._num_anchors,1)
        dims = self._anchor_dims[level-3].repeat(fh, fw, 1, 1)
        anchors = torch.cat([centers, dims], dim=-1)
        return anchors.view(fh*fw*self._num_anchors, 4)

    def get_anchors(self, H, W):
        anchors = []
        for i in range(3,8):
            anchors.append(self._get_anchors(math.ceil(H/2**i), math.ceil(W/2**i), i))
        return torch.cat(anchors, dim=0)

# =========================
# Label Encoder
# =========================
class LabelEncoder:
    def __init__(self):
        self._anchor_box = AnchorBox()
        self._box_variance = torch.tensor(TrainingConfig.BOX_VARIANCE, dtype=torch.float32)

    def _match_anchor_boxes(self, anchors, gt_boxes, match_iou=0.5, ignore_iou=0.4):
        iou_matrix = compute_iou(anchors, gt_boxes)
        max_iou, matched_idx = torch.max(iou_matrix, dim=1)
        positive = max_iou >= match_iou
        negative = max_iou < ignore_iou
        ignore = ~(positive | negative)
        return matched_idx, positive.float(), ignore.float()

    def _compute_box_target(self, anchors, matched_gt):
        # xywh
        box_t = torch.cat([
            (matched_gt[:,:2] - anchors[:,:2]) / anchors[:,2:],
            torch.log(matched_gt[:,2:] / anchors[:,2:])
        ], dim=-1)
        return box_t / self._box_variance

    def _encode_single(self, image_shape, gt_boxes, cls_ids):
        anchors = self._anchor_box.get_anchors(image_shape[2], image_shape[3])
        if gt_boxes.numel() == 0:
            box_target = torch.zeros_like(anchors)
            cls_target = torch.full((anchors.shape[0],), -1.0, dtype=torch.float32)
            return box_target, cls_target

        gt_xywh = convert_to_xywh(gt_boxes)
        matched_idx, positive, ignore = self._match_anchor_boxes(anchors, gt_xywh)
        matched_gt = gt_xywh[matched_idx]
        box_target = self._compute_box_target(anchors, matched_gt)

        cls_ids = cls_ids.float()
        matched_cls = cls_ids[matched_idx]
        cls_target = torch.where(positive!=1.0, -1.0, matched_cls)
        cls_target = torch.where(ignore==1.0, -2.0, cls_target)
        return box_target, cls_target

    def encode_batch(self, images, gt_boxes_list, cls_ids_list):
        box_targets = []
        cls_targets = []
        for i in range(images.size(0)):
            b, c = self._encode_single(images.size(), gt_boxes_list[i], cls_ids_list[i])
            box_targets.append(b); cls_targets.append(c)
        box_targets = torch.stack(box_targets, dim=0)
        cls_targets = torch.stack(cls_targets, dim=0)
        return images, (cls_targets, box_targets)

# =========================
# Model
# =========================
class ResNetBackbone(nn.Module):
    def __init__(self):
        super().__init__()
        resnet = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        self.layer1 = nn.Sequential(*list(resnet.children())[:5])  # stem + layer1
        self.layer2 = resnet.layer2
        self.layer3 = resnet.layer3
        self.layer4 = resnet.layer4

    def forward(self, x):
        c1 = self.layer1(x)
        c2 = self.layer2(c1)
        c3 = self.layer3(c2)
        c4 = self.layer4(c3)
        return c2, c3, c4

class FeaturePyramid(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone
        self.conv_c3_1x1 = nn.Conv2d( 512, 256, 1)
        self.conv_c4_1x1 = nn.Conv2d(1024, 256, 1)
        self.conv_c5_1x1 = nn.Conv2d(2048, 256, 1)
        self.conv_c3_3x3 = nn.Conv2d(256, 256, 3, padding=1)
        self.conv_c4_3x3 = nn.Conv2d(256, 256, 3, padding=1)
        self.conv_c5_3x3 = nn.Conv2d(256, 256, 3, padding=1)
        self.conv_c6_3x3 = nn.Conv2d(2048, 256, 3, stride=2, padding=1)
        self.conv_c7_3x3 = nn.Conv2d(256, 256, 3, stride=2, padding=1)

    def forward(self, x):
        c3, c4, c5 = self.backbone(x)
        p5 = self.conv_c5_1x1(c5)
        p4_lat = self.conv_c4_1x1(c4)
        p4  = p4_lat + F.interpolate(p5, size=p4_lat.shape[2:], mode='nearest')
        p3_lat = self.conv_c3_1x1(c3)
        p3  = p3_lat + F.interpolate(p4, size=p3_lat.shape[2:], mode='nearest')
        p3 = self.conv_c3_3x3(p3); p4 = self.conv_c4_3x3(p4); p5 = self.conv_c5_3x3(p5)
        p6 = self.conv_c6_3x3(c5)
        p7 = self.conv_c7_3x3(F.relu(p6))
        return p3, p4, p5, p6, p7

def build_head(out_filters, bias_init):
    layers = []
    for _ in range(4):
        layers += [nn.Conv2d(256,256,3,padding=1), nn.ReLU()]
    last = nn.Conv2d(256, out_filters, 3, padding=1)
    if isinstance(bias_init, float):
        nn.init.constant_(last.bias, bias_init)
    else:
        nn.init.constant_(last.bias, 0.0)
    layers.append(last)
    return nn.Sequential(*layers)

class RetinaNet(nn.Module):
    def __init__(self, num_classes, backbone=None):
        super().__init__()
        self.num_classes = num_classes
        self.fpn = FeaturePyramid(backbone or ResNetBackbone())
        prior_probability = -torch.log(torch.tensor((1 - 0.01) / 0.01))
        self.cls_head = build_head(9*num_classes, float(prior_probability))
        self.box_head = build_head(9*4, "zeros")

    def forward(self, x):
        feats = self.fpn(x)
        N = x.size(0)
        cls_outs, box_outs = [], []
        for f in feats:
            box_outs.append(self.box_head(f).view(N, -1, 4))
            cls_outs.append(self.cls_head(f).view(N, -1, self.num_classes))
        return torch.cat(cls_outs, 1), torch.cat(box_outs, 1)

# =========================
# Decode & Viz
# =========================
class DecodePredictions(nn.Module):
    def __init__(self):
        super().__init__()
        self.num_classes = TrainingConfig.NUM_CLASSES
        self.confidence_threshold = TrainingConfig.CONFIDENCE_THRESHOLD
        self.nms_iou_threshold = TrainingConfig.NMS_IOU_THRESHOLD
        self.max_detections_per_class = TrainingConfig.MAX_DETECTIONS_PER_CLASS
        self.max_detections = TrainingConfig.MAX_DETECTIONS
        self._anchors = AnchorBox()
        self._box_variance = torch.tensor(TrainingConfig.BOX_VARIANCE, dtype=torch.float32)

    def _decode(self, anchors, box_pred):
        boxes = box_pred * self._box_variance
        boxes = torch.cat([
            boxes[:, :, :2] * anchors[:, :, 2:] + anchors[:, :, :2],
            torch.exp(boxes[:, :, 2:]) * anchors[:, :, 2:]
        ], dim=-1)
        return convert_to_corners(boxes)

    def forward(self, images, preds):
        cls_logits, box_preds = preds
        cls_scores = torch.sigmoid(cls_logits)
        H, W = images.shape[2], images.shape[3]
        anchors = self._anchors.get_anchors(H, W)[None, ...]  # [1, A, 4]
        boxes = self._decode(anchors, box_preds)  # [B, A, 4]

        out_boxes, out_scores, out_classes = [], [], []
        for b in range(images.size(0)):
            bboxes = boxes[b]
            scores = cls_scores[b]  # [A, C]
            for c in range(self.num_classes):
                cs = scores[:, c]
                mask = cs > self.confidence_threshold
                if not mask.any():
                    continue
                c_boxes = bboxes[mask]; c_scores = cs[mask]
                if c_scores.numel() > self.max_detections_per_class:
                    topk = torch.topk(c_scores, self.max_detections_per_class)
                    c_scores = topk.values; c_boxes = c_boxes[topk.indices]
                keep = nms(c_boxes, c_scores, self.nms_iou_threshold)
                out_boxes.append(c_boxes[keep])
                out_scores.append(c_scores[keep])
                out_classes.append(torch.full_like(c_scores[keep], c, dtype=torch.int64))
        if not out_boxes:
            return torch.empty(0,4), torch.empty(0), torch.empty(0, dtype=torch.int64)
        fb = torch.cat(out_boxes,0); fs = torch.cat(out_scores,0); fc = torch.cat(out_classes,0)
        if fs.numel() > self.max_detections:
            topk = torch.topk(fs, self.max_detections)
            fb, fs, fc = fb[topk.indices], topk.values, fc[topk.indices]
        return fb, fs, fc

def visualize_detections(image, boxes, classes, scores, figsize=(7,7), linewidth=1):
    img = np.array(image, dtype=np.uint8)
    plt.figure(figsize=figsize); plt.axis("off"); plt.imshow(img)
    ax = plt.gca()
    for box, c, s in zip(boxes, classes, scores):
        x1,y1,x2,y2 = box
        w,h = x2-x1, y2-y1
        rect = plt.Rectangle((x1,y1), w,h, fill=False, linewidth=linewidth)
        ax.add_patch(rect)
        ax.text(x1, y1, f"{int(c)}:{s:.2f}", bbox={"facecolor":"white","alpha":0.5}, fontsize=8)
    plt.show()

# =========================
# Loss
# =========================
class RetinaNetBoxLoss(nn.Module):
    def __init__(self, delta):
        super().__init__()
        self.delta = delta
    def forward(self, y_true, y_pred):
        d = y_true - y_pred
        ad = torch.abs(d)
        loss = torch.where(ad < self.delta, 0.5 * (d**2), ad - 0.5)
        return torch.sum(loss, dim=-1)

class RetinaNetClassificationLoss(nn.Module):
    def __init__(self, alpha, gamma):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
    def forward(self, y_true, y_pred):
        ce = F.binary_cross_entropy_with_logits(y_pred, y_true, reduction='none')
        p  = torch.sigmoid(y_pred)
        alpha = torch.where(y_true == 1.0, self.alpha, 1.0 - self.alpha)
        pt = torch.where(y_true == 1.0, p, 1.0 - p)
        loss = alpha * (1.0 - pt) ** self.gamma * ce
        return torch.sum(loss, dim=-1)

class RetinaNetLoss(nn.Module):
    def __init__(self, num_classes=TrainingConfig.NUM_CLASSES):
        super().__init__()
        self.num_classes = num_classes
        self.clf = RetinaNetClassificationLoss(TrainingConfig.CLASSIFICATION_LOSS_ALPHA,
                                               TrainingConfig.CLASSIFICATION_LOSS_GAMMA)
        self.box = RetinaNetBoxLoss(TrainingConfig.BOX_LOSS_DELTA)
    def forward(self, preds, targets):
        cls_logits, box_preds = preds
        cls_targets, box_targets = targets
        positive_mask = (cls_targets > -1.0).float()
        ignore_mask   = (cls_targets == -2.0).float()
        normalizer = torch.sum(positive_mask, dim=-1).clamp(min=1.0)

        temp = torch.where(cls_targets >= 0, cls_targets.long(), 0)
        one_hot = F.one_hot(temp, num_classes=self.num_classes).float()
        one_hot = one_hot * positive_mask.unsqueeze(-1)

        clf_loss = self.clf(one_hot, cls_logits.float())
        clf_loss = torch.where(ignore_mask==1.0, 0.0, clf_loss)
        clf_loss = torch.sum(clf_loss, dim=-1) / normalizer

        box_loss = self.box(box_targets.float(), box_preds.float())
        box_loss = torch.where(positive_mask==1.0, box_loss, 0.0)
        box_loss = torch.sum(box_loss, dim=-1) / normalizer

        return clf_loss + box_loss

# =========================
# Transform
# =========================
IMG_SIZE = 512  # 384~512 권장

pil_to_tensor = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),   # 고정 리사이즈로 피처맵/앵커 수 감소
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406],
                         std =[0.229,0.224,0.225]),
])

# =========================
# DataLoader
# =========================
train_ds = TFRecordKITTIDataset(config.TRAIN_TFRECORD_PATTERN, transform=pil_to_tensor, use_multi=True)
val_ds   = TFRecordKITTIDataset(config.VALIDATION_TFRECORD_PATTERN, transform=pil_to_tensor, use_multi=True)

train_loader = DataLoader(train_ds, batch_size=config.BATCH_SIZE, collate_fn=collate_fn, num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=config.BATCH_SIZE, collate_fn=collate_fn, num_workers=0)

# =========================
# Model / Optim / Sched
# =========================
model = RetinaNet(config.NUM_CLASSES).to(device)
label_encoder = LabelEncoder()
loss_fn = RetinaNetLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=config.LEARNING_RATES[0], momentum=config.OPTIMIZER_MOMENTUM)

def lr_lambda(epoch):
    for i,b in enumerate(config.LEARNING_RATE_BOUNDARIES):
        if epoch < b:
            return config.LEARNING_RATES[i] / config.LEARNING_RATES[0]
    return config.LEARNING_RATES[-1] / config.LEARNING_RATES[0]

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

# =========================
# Train / Eval (AMP 최신 API 적용)
# =========================
from torch import amp

scaler = amp.GradScaler('cuda', enabled=torch.cuda.is_available())

start_epoch, best_val = 0, float('inf')

if os.path.exists(config.LATEST_CHECKPOINT_FILE):
    ckpt = torch.load(config.LATEST_CHECKPOINT_FILE, map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    start_epoch = ckpt['epoch'] + 1
    print(f"[Resume] start from epoch {start_epoch}")

for epoch in range(start_epoch, config.NUM_EPOCHS):
    t0 = time.time()
    model.train()
    tr_loss = 0.0
    tr_batches = 0

    for i, (imgs, gt_boxes, cls_ids) in enumerate(train_loader):
        imgs = imgs.to(device, non_blocking=True)

        # encode는 CPU 텐서 목록이므로, 결과만 GPU로 보냅니다.
        imgs, (cls_t, box_t) = label_encoder.encode_batch(imgs, gt_boxes, cls_ids)
        cls_t, box_t = cls_t.to(device, non_blocking=True), box_t.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        # AMP (최신 API)
        with amp.autocast('cuda', enabled=torch.cuda.is_available()):
            pred = model(imgs)
            loss = torch.mean(loss_fn(pred, (cls_t, box_t)))

        # AMP 스케일링
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        tr_loss += float(loss.detach().item())
        tr_batches += 1

        if (i+1) % 10 == 0:
            print(f"Epoch {epoch+1}/{config.NUM_EPOCHS} | Step {i+1} | loss {loss.item():.4f}")

        # 메모리 단편화 완화(선택)
        del imgs, cls_t, box_t, pred, loss
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    tr_loss /= max(1, tr_batches)

    # ===== Validation =====
    model.eval()
    va_loss = 0.0
    va_batches = 0
    with torch.no_grad(), amp.autocast('cuda', enabled=torch.cuda.is_available()):
        for imgs, gt_boxes, cls_ids in val_loader:
            imgs = imgs.to(device, non_blocking=True)
            imgs, (cls_t, box_t) = label_encoder.encode_batch(imgs, gt_boxes, cls_ids)
            cls_t, box_t = cls_t.to(device, non_blocking=True), box_t.to(device, non_blocking=True)

            pred = model(imgs)
            loss = torch.mean(loss_fn(pred, (cls_t, box_t)))
            va_loss += float(loss.detach().item())
            va_batches += 1

            del imgs, cls_t, box_t, pred, loss
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    va_loss /= max(1, va_batches)
    dt = time.time() - t0
    print(f"[Epoch {epoch+1}] train {tr_loss:.4f} | val {va_loss:.4f} | {dt:.1f}s")

    scheduler.step()

    # latest
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict()
    }, config.LATEST_CHECKPOINT_FILE)

    # per-epoch
    torch.save({'epoch': epoch, 'model_state_dict': model.state_dict()},
               config.EPOCH_CHECKPOINT_TEMPLATE.format(epoch=epoch+1))

    if va_loss < best_val:
        best_val = va_loss
        torch.save({'epoch': epoch, 'model_state_dict': model.state_dict()}, config.BEST_CHECKPOINT_FILE)
        print(f"** New Best Saved (val {va_loss:.4f}) **")

# =========================
# Test (체크포인트들 평가)
# =========================
test_ds = TFRecordKITTIDataset(config.TEST_TFRECORD_PATTERN, transform=pil_to_tensor, use_multi=True)

def evaluate_ckpt_on_test(model, ckpt_path):
    # 매 평가마다 새로운 DataLoader 생성 (IterableDataset 재사용 이슈 방지)
    test_loader = DataLoader(test_ds, batch_size=config.BATCH_SIZE, collate_fn=collate_fn, num_workers=0)
    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()
    total = 0.0
    n_batches = 0
    with torch.no_grad(), amp.autocast('cuda', enabled=torch.cuda.is_available()):
        for imgs, gt_boxes, cls_ids in test_loader:
            imgs = imgs.to(device)
            imgs, (cls_t, box_t) = label_encoder.encode_batch(imgs, gt_boxes, cls_ids)
            cls_t, box_t = cls_t.to(device), box_t.to(device)
            pred = model(imgs)
            loss = torch.mean(loss_fn(pred, (cls_t, box_t)))
            total += float(loss.detach().item())
            n_batches += 1
    avg = total / max(1, n_batches)
    print(f"{os.path.basename(ckpt_path)} | test loss {avg:.4f}")

ckpt_paths = sorted(
    glob.glob(config.EPOCH_CHECKPOINT_TEMPLATE.format(epoch='*')),
    key=lambda x: int(re.search(r'checkpoint_epoch_(\d+).pth', x).group(1))
)

if not ckpt_paths:
    print("No epoch checkpoints found.")
else:
    for p in ckpt_paths:
        evaluate_ckpt_on_test(model, p)


Python : 3.12.11 | packaged by conda-forge | (main, Jun  4 2025, 14:45:31) [GCC 13.3.0]
PyTorch: 2.7.1+cu118
CUDA   : 11.8 | is_available: True
[Info] use_multi=True가 전달되었지만, read-only/캐시 인덱스 호환을 위해 파일별 TFRecordDataset 체인 방식으로 대체합니다.
[Info] use_multi=True가 전달되었지만, read-only/캐시 인덱스 호환을 위해 파일별 TFRecordDataset 체인 방식으로 대체합니다.
Epoch 1/10 | Step 10 | loss 0.2972
Epoch 1/10 | Step 20 | loss 0.2972
Epoch 1/10 | Step 30 | loss 0.2972
Epoch 1/10 | Step 40 | loss 0.2972
Epoch 1/10 | Step 50 | loss 0.2972
Epoch 1/10 | Step 60 | loss 0.2972
Epoch 1/10 | Step 70 | loss 0.2972
Epoch 1/10 | Step 80 | loss 0.2972
Epoch 1/10 | Step 90 | loss 0.2972
Epoch 1/10 | Step 100 | loss 0.2972
Epoch 1/10 | Step 110 | loss 0.2972
Epoch 1/10 | Step 120 | loss 0.2971
Epoch 1/10 | Step 130 | loss 0.2971
Epoch 1/10 | Step 140 | loss 0.2971
Epoch 1/10 | Step 150 | loss 0.2971
Epoch 1/10 | Step 160 | loss 0.2971
Epoch 1/10 | Step 170 | loss 0.2971
Epoch 1/10 | Step 180 | loss 0.2971
Epoch 1/10 | Step 190 | loss 0.2971
E

KeyboardInterrupt: 

In [8]:
# !pip install -q tfrecord pillow matplotlib torchvision --upgrade

import sys, os, torch
print("Python :", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA   :", torch.version.cuda, "| is_available:", torch.cuda.is_available())

import os, io, re, glob, time, math, itertools, copy, hashlib
from typing import Dict, Any, List, Tuple

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, IterableDataset
from torchvision import transforms, models
from torchvision.ops import nms

# =========================
# Config
# =========================
class TrainingConfig:
    ROOT_DIR = os.path.join(os.getenv("HOME") or "", "work/object_detection")

    # Data
    DATA_PATH = os.path.join(ROOT_DIR, "data")
    TRAIN_TFRECORD_PATTERN      = os.path.join(DATA_PATH, "kitti/3.2.0/kitti-train.tfrecord-*")
    VALIDATION_TFRECORD_PATTERN = os.path.join(DATA_PATH, "kitti/3.2.0/kitti-validation.tfrecord-*")
    TEST_TFRECORD_PATTERN       = os.path.join(DATA_PATH, "kitti/3.2.0/kitti-test.tfrecord-*")

    # Model
    NUM_CLASSES = 8

    # Training  (OOM 방지 기본값 소량 권장)
    BATCH_SIZE = 2
    NUM_EPOCHS = 10
    LEARNING_RATES = [2.5e-03, 2.5e-03, 2.5e-04, 2.5e-04, 2.5e-04, 2.5e-04]
    LEARNING_RATE_BOUNDARIES = [125, 250, 500, 240000, 360000]  # epoch 기준 (단, 여기선 람다 스케줄러로 사용)
    OPTIMIZER_MOMENTUM = 0.9

    # Loss parameters
    CLASSIFICATION_LOSS_ALPHA = 0.25
    CLASSIFICATION_LOSS_GAMMA = 2.0
    BOX_LOSS_DELTA = 1.0

    # Decoder parameters
    CONFIDENCE_THRESHOLD = 0.05
    NMS_IOU_THRESHOLD = 0.5
    MAX_DETECTIONS_PER_CLASS = 100
    MAX_DETECTIONS = 100
    BOX_VARIANCE = [0.1, 0.1, 0.2, 0.2]

    # Checkpoint
    CHECKPOINT_PATH = os.path.join(ROOT_DIR, 'checkpoints')
    LATEST_CHECKPOINT_FILE = os.path.join(CHECKPOINT_PATH, 'latest_checkpoint.pth')
    BEST_CHECKPOINT_FILE   = os.path.join(CHECKPOINT_PATH, 'best_checkpoint.pth')
    EPOCH_CHECKPOINT_TEMPLATE = os.path.join(CHECKPOINT_PATH, 'checkpoint_epoch_{epoch}.pth')

config = TrainingConfig()
os.makedirs(config.CHECKPOINT_PATH, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

# =========================
# TFRecord helpers
# =========================
try:
    from tfrecord.torch.dataset import TFRecordDataset, MultiTFRecordDataset
except ModuleNotFoundError:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "tfrecord"])
    from tfrecord.torch.dataset import TFRecordDataset, MultiTFRecordDataset

def _index_cache_dir() -> str:
    """index 파일 캐시 디렉터리: ~/work/object_detection/index_cache"""
    root = os.path.join(os.getenv("HOME") or "", "work/object_detection", "index_cache")
    os.makedirs(root, exist_ok=True)
    return root

def _hashed_index_path(tfrecord_path: str) -> str:
    """원본 경로 해시 기반 index 파일명 생성"""
    h = hashlib.sha1(tfrecord_path.encode("utf-8")).hexdigest()[:16]
    base = os.path.basename(tfrecord_path)
    return os.path.join(_index_cache_dir(), f"{base}.{h}.index")

def ensure_index_file(data_path: str) -> str:
    """
    캐시에 index가 없거나, 크기가 비정상(빈 파일/깨짐)이면 재생성.
    생성 시 임시 파일로 만든 뒤 원자적 교체(os.replace)로 안전하게 기록.
    """
    from tfrecord.tools.tfrecord2idx import create_index

    cache_idx = _hashed_index_path(data_path)
    def _bad(p: str) -> bool:
        return (not os.path.exists(p)) or (os.path.getsize(p) < 16)  # 아주 작은 파일은 손상으로 간주

    if _bad(cache_idx):
        tmp_path = cache_idx + ".tmp"
        if os.path.exists(tmp_path):
            try:
                os.remove(tmp_path)
            except OSError:
                pass
        create_index(data_path, tmp_path)   # 생성
        # 생성 실패로 tmp가 너무 작으면 기존 것도 쓰지 않고 예외를 던지게 둠
        os.replace(tmp_path, cache_idx)     # 원자적 교체
    return cache_idx

# =========================
# Dataset / Collate
# =========================
class TFRecordKITTIDataset(IterableDataset):
    def __init__(self, tfrecord_pattern: str, transform=None, use_multi=False):
        super().__init__()
        self.user_transform = transform
        self.file_paths = sorted([p for p in glob.glob(tfrecord_pattern) if not p.endswith('.index')])
        if not self.file_paths:
            raise FileNotFoundError(f"No TFRecord files found for pattern: {tfrecord_pattern}")

        self.feature_description = {
            'image': "byte", 'image/file_name': "byte",
            'objects/bbox': "float", 'objects/type': "int",
        }

        # 항상 체인 방식 사용 (read-only + 캐시 인덱스와 호환)
        if use_multi:
            print("[Info] use_multi=True가 전달되었지만, read-only/캐시 인덱스 호환을 위해 "
                  "파일별 TFRecordDataset 체인 방식으로 대체합니다.")
        self.datasets = []
        for p in self.file_paths:
            idx = ensure_index_file(p)  # 캐시 디렉터리에 .index 생성/검증
            ds = TFRecordDataset(
                data_path=p,
                index_path=idx,
                description=self.feature_description,
                transform=self._parse_record,
                shuffle_queue_size=1024  # 가능하면 셔플 버퍼 사용
            )
            self.datasets.append(ds)

    def _parse_record(self, features: Dict[str, Any]) -> Dict[str, Any]:
        # image
        image = Image.open(io.BytesIO(features['image'])).convert('RGB')

        # filename: bytes 또는 str 모두 안전 처리
        fname_raw = features.get('image/file_name', b'')
        if isinstance(fname_raw, bytes):
            filename = fname_raw.decode('utf-8', errors='ignore')
        else:
            filename = str(fname_raw)

        # bboxes
        b_raw = features.get('objects/bbox')
        if b_raw is not None and len(b_raw) > 0:
            b = torch.tensor(b_raw, dtype=torch.float32).view(-1, 4)
            # (ymin, xmin, ymax, xmax) -> (xmin, ymin, xmax, ymax)
            b = b[:, [1, 0, 3, 2]]
        else:
            b = torch.empty((0, 4), dtype=torch.float32)

        # labels
        t_raw = features.get('objects/type')
        if t_raw is not None and len(t_raw) > 0:
            labels = torch.tensor(t_raw, dtype=torch.int64)
        else:
            labels = torch.empty((0,), dtype=torch.int64)

        sample = {'image': image, 'bbox': b, 'class_id': labels, 'filename': filename}
        if self.user_transform is not None:
            # transform은 보통 이미지(Tensor)만 변환
            sample['image'] = self.user_transform(sample['image'])
        return sample

    def __iter__(self):
        return itertools.chain.from_iterable(self.datasets)

def collate_fn(batch: List[Dict[str, Any]]) -> Tuple[torch.Tensor, List[torch.Tensor], List[torch.Tensor]]:
    images = [item['image'] for item in batch]
    bboxes = [item['bbox'] for item in batch]
    class_ids = [item['class_id'] for item in batch]

    for i, img in enumerate(images):
        if not torch.is_tensor(img) or img.dim() != 3:
            raise TypeError(f"Image at {i} is not Tensor[C,H,W]: {type(img)}, shape={getattr(img, 'shape', None)}")

    C = images[0].shape[0]
    max_h = max(int(img.shape[1]) for img in images)
    max_w = max(int(img.shape[2]) for img in images)

    padded = torch.zeros(len(images), C, max_h, max_w, dtype=images[0].dtype)
    for i, img in enumerate(images):
        _, h, w = img.shape
        padded[i, :, :h, :w] = img
    return padded, bboxes, class_ids

# =========================
# Geometry / Anchors
# =========================
def swap_xy(boxes):  # [x,y swap]
    return torch.stack([boxes[:,1], boxes[:,0], boxes[:,3], boxes[:,2]], dim=-1)

def convert_to_xywh(boxes):
    return torch.cat([(boxes[...,:2] + boxes[...,2:]) / 2.0,
                      boxes[...,2:] - boxes[...,:2]], dim=-1)

def convert_to_corners(boxes):
    return torch.cat([boxes[...,:2] - boxes[...,2:]/2.0,
                      boxes[...,:2] + boxes[...,2:]/2.0], dim=-1)

def compute_iou(boxes1, boxes2):
    b1 = convert_to_corners(boxes1)
    b2 = convert_to_corners(boxes2)
    lu = torch.maximum(b1[:,None,:2], b2[:,:2])
    rd = torch.minimum(b1[:,None,2:], b2[:,2:])
    inter = torch.clamp(rd - lu, min=0.0)
    inter_area = inter[:,:,0] * inter[:,:,1]
    a1 = boxes1[:,2] * boxes1[:,3]
    a2 = boxes2[:,2] * boxes2[:,3]
    union = torch.clamp(a1[:,None] + a2 - inter_area, min=1e-8)
    return torch.clamp(inter_area / union, 0.0, 1.0)

class AnchorBox:
    def __init__(self):
        self.aspect_ratios = [0.5, 1.0, 2.0]
        self.scales = [2 ** x for x in [0, 1/3, 2/3]]
        self._num_anchors = len(self.aspect_ratios) * len(self.scales)
        self._strides = [2 ** i for i in range(3, 8)]
        self._areas = [x**2 for x in [32.0, 64.0, 128.0, 256.0, 512.0]]
        self._anchor_dims = self._compute_dims()

    def _compute_dims(self):
        out = []
        for area in self._areas:
            dims = []
            for r in self.aspect_ratios:
                h = torch.sqrt(torch.tensor(area / r))
                w = area / h
                for s in self.scales:
                    dims.append(torch.tensor([w*s, h*s]))
            out.append(torch.stack(dims, dim=0).view(1,1,self._num_anchors,2))
        return out

    def _get_anchors(self, fh, fw, level):
        rx = torch.arange(fw, dtype=torch.float32) + 0.5
        ry = torch.arange(fh, dtype=torch.float32) + 0.5
        centers = torch.stack(torch.meshgrid(rx, ry, indexing='xy'), dim=-1) * self._strides[level-3]
        centers = centers.unsqueeze(-2).repeat(1,1,self._num_anchors,1)
        dims = self._anchor_dims[level-3].repeat(fh, fw, 1, 1)
        anchors = torch.cat([centers, dims], dim=-1)
        return anchors.view(fh*fw*self._num_anchors, 4)

    def get_anchors(self, H, W):
        anchors = []
        for i in range(3,8):
            anchors.append(self._get_anchors(math.ceil(H/2**i), math.ceil(W/2**i), i))
        return torch.cat(anchors, dim=0)

# =========================
# Label Encoder
# =========================
class LabelEncoder:
    def __init__(self):
        self._anchor_box = AnchorBox()
        self._box_variance = torch.tensor(TrainingConfig.BOX_VARIANCE, dtype=torch.float32)

    def _match_anchor_boxes(self, anchors, gt_boxes, match_iou=0.5, ignore_iou=0.4):
        iou_matrix = compute_iou(anchors, gt_boxes)
        max_iou, matched_idx = torch.max(iou_matrix, dim=1)
        positive = max_iou >= match_iou
        negative = max_iou < ignore_iou
        ignore = ~(positive | negative)
        return matched_idx, positive.float(), ignore.float()

    def _compute_box_target(self, anchors, matched_gt):
        # xywh
        box_t = torch.cat([
            (matched_gt[:,:2] - anchors[:,:2]) / anchors[:,2:],
            torch.log(matched_gt[:,2:] / anchors[:,2:])
        ], dim=-1)
        return box_t / self._box_variance

    def _encode_single(self, image_shape, gt_boxes, cls_ids):
        anchors = self._anchor_box.get_anchors(image_shape[2], image_shape[3])
        if gt_boxes.numel() == 0:
            box_target = torch.zeros_like(anchors)
            cls_target = torch.full((anchors.shape[0],), -1.0, dtype=torch.float32)
            return box_target, cls_target

        gt_xywh = convert_to_xywh(gt_boxes)
        matched_idx, positive, ignore = self._match_anchor_boxes(anchors, gt_xywh)
        matched_gt = gt_xywh[matched_idx]
        box_target = self._compute_box_target(anchors, matched_gt)

        cls_ids = cls_ids.float()
        matched_cls = cls_ids[matched_idx]
        cls_target = torch.where(positive!=1.0, -1.0, matched_cls)
        cls_target = torch.where(ignore==1.0, -2.0, cls_target)
        return box_target, cls_target

    def encode_batch(self, images, gt_boxes_list, cls_ids_list):
        box_targets = []
        cls_targets = []
        for i in range(images.size(0)):
            b, c = self._encode_single(images.size(), gt_boxes_list[i], cls_ids_list[i])
            box_targets.append(b); cls_targets.append(c)
        box_targets = torch.stack(box_targets, dim=0)
        cls_targets = torch.stack(cls_targets, dim=0)
        return images, (cls_targets, box_targets)

# =========================
# Model
# =========================
class ResNetBackbone(nn.Module):
    def __init__(self):
        super().__init__()
        resnet = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        self.layer1 = nn.Sequential(*list(resnet.children())[:5])  # stem + layer1
        self.layer2 = resnet.layer2
        self.layer3 = resnet.layer3
        self.layer4 = resnet.layer4

    def forward(self, x):
        c1 = self.layer1(x)
        c2 = self.layer2(c1)
        c3 = self.layer3(c2)
        c4 = self.layer4(c3)
        return c2, c3, c4

class FeaturePyramid(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone
        self.conv_c3_1x1 = nn.Conv2d( 512, 256, 1)
        self.conv_c4_1x1 = nn.Conv2d(1024, 256, 1)
        self.conv_c5_1x1 = nn.Conv2d(2048, 256, 1)
        self.conv_c3_3x3 = nn.Conv2d(256, 256, 3, padding=1)
        self.conv_c4_3x3 = nn.Conv2d(256, 256, 3, padding=1)
        self.conv_c5_3x3 = nn.Conv2d(256, 256, 3, padding=1)
        self.conv_c6_3x3 = nn.Conv2d(2048, 256, 3, stride=2, padding=1)
        self.conv_c7_3x3 = nn.Conv2d(256, 256, 3, stride=2, padding=1)

    def forward(self, x):
        c3, c4, c5 = self.backbone(x)
        p5 = self.conv_c5_1x1(c5)
        p4_lat = self.conv_c4_1x1(c4)
        p4  = p4_lat + F.interpolate(p5, size=p4_lat.shape[2:], mode='nearest')
        p3_lat = self.conv_c3_1x1(c3)
        p3  = p3_lat + F.interpolate(p4, size=p3_lat.shape[2:], mode='nearest')
        p3 = self.conv_c3_3x3(p3); p4 = self.conv_c4_3x3(p4); p5 = self.conv_c5_3x3(p5)
        p6 = self.conv_c6_3x3(c5)
        p7 = self.conv_c7_3x3(F.relu(p6))
        return p3, p4, p5, p6, p7

def build_head(out_filters, bias_init):
    layers = []
    for _ in range(4):
        layers += [nn.Conv2d(256,256,3,padding=1), nn.ReLU()]
    last = nn.Conv2d(256, out_filters, 3, padding=1)
    if isinstance(bias_init, float):
        nn.init.constant_(last.bias, bias_init)
    else:
        nn.init.constant_(last.bias, 0.0)
    layers.append(last)
    return nn.Sequential(*layers)

class RetinaNet(nn.Module):
    def __init__(self, num_classes, backbone=None):
        super().__init__()
        self.num_classes = num_classes
        self.fpn = FeaturePyramid(backbone or ResNetBackbone())
        prior_probability = -torch.log(torch.tensor((1 - 0.01) / 0.01))
        self.cls_head = build_head(9*num_classes, float(prior_probability))
        self.box_head = build_head(9*4, "zeros")

    def forward(self, x):
        feats = self.fpn(x)
        N = x.size(0)
        cls_outs, box_outs = [], []
        for f in feats:
            box_outs.append(self.box_head(f).view(N, -1, 4))
            cls_outs.append(self.cls_head(f).view(N, -1, self.num_classes))
        return torch.cat(cls_outs, 1), torch.cat(box_outs, 1)

# =========================
# Decode & Viz
# =========================
class DecodePredictions(nn.Module):
    def __init__(self):
        super().__init__()
        self.num_classes = TrainingConfig.NUM_CLASSES
        self.confidence_threshold = TrainingConfig.CONFIDENCE_THRESHOLD
        self.nms_iou_threshold = TrainingConfig.NMS_IOU_THRESHOLD
        self.max_detections_per_class = TrainingConfig.MAX_DETECTIONS_PER_CLASS
        self.max_detections = TrainingConfig.MAX_DETECTIONS
        self._anchors = AnchorBox()
        # ✅ buffer로 등록하여 module.to(device) 시 함께 이동
        self.register_buffer("_box_variance",
                             torch.tensor(TrainingConfig.BOX_VARIANCE, dtype=torch.float32))

    def _decode(self, anchors, box_pred):
        # box_pred와 같은 디바이스에서 연산
        boxes = box_pred * self._box_variance
        boxes = torch.cat([
            boxes[:, :, :2] * anchors[:, :, 2:] + anchors[:, :, :2],
            torch.exp(boxes[:, :, 2:]) * anchors[:, :, 2:]
        ], dim=-1)
        return convert_to_corners(boxes)

    def forward(self, images, preds):
        cls_logits, box_preds = preds
        cls_scores = torch.sigmoid(cls_logits)
        H, W = images.shape[2], images.shape[3]
        anchors = self._anchors.get_anchors(H, W)[None, ...]
        anchors = anchors.to(box_preds.device)  # ✅ 디바이스 일치
        boxes = self._decode(anchors, box_preds)

        out_boxes, out_scores, out_classes = [], [], []
        for b in range(images.size(0)):
            bboxes = boxes[b]
            scores = cls_scores[b]  # [A, C]
            for c in range(self.num_classes):
                cs = scores[:, c]
                mask = cs > self.confidence_threshold
                if not mask.any():
                    continue
                c_boxes = bboxes[mask]; c_scores = cs[mask]
                if c_scores.numel() > self.max_detections_per_class:
                    topk = torch.topk(c_scores, self.max_detections_per_class)
                    c_scores = topk.values; c_boxes = c_boxes[topk.indices]
                keep = nms(c_boxes, c_scores, self.nms_iou_threshold)
                out_boxes.append(c_boxes[keep])
                out_scores.append(c_scores[keep])
                out_classes.append(torch.full_like(c_scores[keep], c, dtype=torch.int64))
        if not out_boxes:
            dev = box_preds.device
            return (torch.empty(0,4, device=dev),
                    torch.empty(0, device=dev),
                    torch.empty(0, dtype=torch.int64, device=dev))
        fb = torch.cat(out_boxes,0); fs = torch.cat(out_scores,0); fc = torch.cat(out_classes,0)
        if fs.numel() > self.max_detections:
            topk = torch.topk(fs, self.max_detections)
            fb, fs, fc = fb[topk.indices], topk.values, fc[topk.indices]
        return fb, fs, fc

def visualize_detections(image, boxes, classes, scores, figsize=(7,7), linewidth=1):
    img = np.array(image, dtype=np.uint8)
    plt.figure(figsize=figsize); plt.axis("off"); plt.imshow(img)
    ax = plt.gca()
    for box, c, s in zip(boxes, classes, scores):
        x1,y1,x2,y2 = box
        w,h = x2-x1, y2-y1
        rect = plt.Rectangle((x1,y1), w,h, fill=False, linewidth=linewidth)
        ax.add_patch(rect)
        ax.text(x1, y1, f"{int(c)}:{s:.2f}", bbox={"facecolor":"white","alpha":0.5}, fontsize=8)
    plt.show()

# =========================
# Loss
# =========================
class RetinaNetBoxLoss(nn.Module):
    def __init__(self, delta):
        super().__init__()
        self.delta = delta
    def forward(self, y_true, y_pred):
        d = y_true - y_pred
        ad = torch.abs(d)
        loss = torch.where(ad < self.delta, 0.5 * (d**2), ad - 0.5)
        return torch.sum(loss, dim=-1)

class RetinaNetClassificationLoss(nn.Module):
    def __init__(self, alpha, gamma):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
    def forward(self, y_true, y_pred):
        ce = F.binary_cross_entropy_with_logits(y_pred, y_true, reduction='none')
        p  = torch.sigmoid(y_pred)
        alpha = torch.where(y_true == 1.0, self.alpha, 1.0 - self.alpha)
        pt = torch.where(y_true == 1.0, p, 1.0 - p)
        loss = alpha * (1.0 - pt) ** self.gamma * ce
        return torch.sum(loss, dim=-1)

class RetinaNetLoss(nn.Module):
    def __init__(self, num_classes=TrainingConfig.NUM_CLASSES):
        super().__init__()
        self.num_classes = num_classes
        self.clf = RetinaNetClassificationLoss(TrainingConfig.CLASSIFICATION_LOSS_ALPHA,
                                               TrainingConfig.CLASSIFICATION_LOSS_GAMMA)
        self.box = RetinaNetBoxLoss(TrainingConfig.BOX_LOSS_DELTA)
    def forward(self, preds, targets):
        cls_logits, box_preds = preds
        cls_targets, box_targets = targets
        positive_mask = (cls_targets > -1.0).float()
        ignore_mask   = (cls_targets == -2.0).float()
        normalizer = torch.sum(positive_mask, dim=-1).clamp(min=1.0)

        temp = torch.where(cls_targets >= 0, cls_targets.long(), 0)
        one_hot = F.one_hot(temp, num_classes=self.num_classes).float()
        one_hot = one_hot * positive_mask.unsqueeze(-1)

        clf_loss = self.clf(one_hot, cls_logits.float())
        clf_loss = torch.where(ignore_mask==1.0, 0.0, clf_loss)
        clf_loss = torch.sum(clf_loss, dim=-1) / normalizer

        box_loss = self.box(box_targets.float(), box_preds.float())
        box_loss = torch.where(positive_mask==1.0, box_loss, 0.0)
        box_loss = torch.sum(box_loss, dim=-1) / normalizer

        return clf_loss + box_loss

# =========================
# Transform
# =========================
IMG_SIZE = 512  # 384~512 권장

pil_to_tensor = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),   # 고정 리사이즈로 피처맵/앵커 수 감소
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406],
                         std =[0.229,0.224,0.225]),
])

# =========================
# DataLoader
# =========================
train_ds = TFRecordKITTIDataset(config.TRAIN_TFRECORD_PATTERN, transform=pil_to_tensor, use_multi=True)
val_ds   = TFRecordKITTIDataset(config.VALIDATION_TFRECORD_PATTERN, transform=pil_to_tensor, use_multi=True)

train_loader = DataLoader(train_ds, batch_size=config.BATCH_SIZE, collate_fn=collate_fn, num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=config.BATCH_SIZE, collate_fn=collate_fn, num_workers=0)

# =========================
# Model / Optim / Sched
# =========================
model = RetinaNet(config.NUM_CLASSES).to(device)
label_encoder = LabelEncoder()
loss_fn = RetinaNetLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=config.LEARNING_RATES[0], momentum=config.OPTIMIZER_MOMENTUM)

def lr_lambda(epoch):
    for i,b in enumerate(config.LEARNING_RATE_BOUNDARIES):
        if epoch < b:
            return config.LEARNING_RATES[i] / config.LEARNING_RATES[0]
    return config.LEARNING_RATES[-1] / config.LEARNING_RATES[0]

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

# =========================
# Train / Eval (AMP 최신 API 적용)
# =========================
from torch import amp
scaler = amp.GradScaler('cuda', enabled=torch.cuda.is_available())

start_epoch, best_val = 0, float('inf')

if os.path.exists(config.LATEST_CHECKPOINT_FILE):
    ckpt = torch.load(config.LATEST_CHECKPOINT_FILE, map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    start_epoch = ckpt['epoch'] + 1
    print(f"[Resume] start from epoch {start_epoch}")

for epoch in range(start_epoch, config.NUM_EPOCHS):
    t0 = time.time()
    model.train()
    tr_loss = 0.0
    tr_batches = 0

    for i, (imgs, gt_boxes, cls_ids) in enumerate(train_loader):
        imgs = imgs.to(device, non_blocking=True)

        # encode는 CPU 텐서 목록이므로, 결과만 GPU로 보냅니다.
        imgs, (cls_t, box_t) = label_encoder.encode_batch(imgs, gt_boxes, cls_ids)
        cls_t, box_t = cls_t.to(device, non_blocking=True), box_t.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        # AMP (최신 API)
        with amp.autocast('cuda', enabled=torch.cuda.is_available()):
            pred = model(imgs)
            loss = torch.mean(loss_fn(pred, (cls_t, box_t)))

        # AMP 스케일링
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        tr_loss += float(loss.detach().item())
        tr_batches += 1

        if (i+1) % 10 == 0:
            print(f"Epoch {epoch+1}/{config.NUM_EPOCHS} | Step {i+1} | loss {loss.item():.4f}")

        # 메모리 단편화 완화(선택)
        del imgs, cls_t, box_t, pred, loss
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    tr_loss /= max(1, tr_batches)

    # ===== Validation =====
    model.eval()
    va_loss = 0.0
    va_batches = 0
    with torch.no_grad(), amp.autocast('cuda', enabled=torch.cuda.is_available()):
        for imgs, gt_boxes, cls_ids in val_loader:
            imgs = imgs.to(device, non_blocking=True)
            imgs, (cls_t, box_t) = label_encoder.encode_batch(imgs, gt_boxes, cls_ids)
            cls_t, box_t = cls_t.to(device, non_blocking=True), box_t.to(device, non_blocking=True)

            pred = model(imgs)
            loss = torch.mean(loss_fn(pred, (cls_t, box_t)))
            va_loss += float(loss.detach().item())
            va_batches += 1

            del imgs, cls_t, box_t, pred, loss
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    va_loss /= max(1, va_batches)
    dt = time.time() - t0
    print(f"[Epoch {epoch+1}] train {tr_loss:.4f} | val {va_loss:.4f} | {dt:.1f}s")

    scheduler.step()

    # latest
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict()
    }, config.LATEST_CHECKPOINT_FILE)

    # per-epoch
    torch.save({'epoch': epoch, 'model_state_dict': model.state_dict()},
               config.EPOCH_CHECKPOINT_TEMPLATE.format(epoch=epoch+1))

    if va_loss < best_val:
        best_val = va_loss
        torch.save({'epoch': epoch, 'model_state_dict': model.state_dict()}, config.BEST_CHECKPOINT_FILE)
        print(f"** New Best Saved (val {va_loss:.4f}) **")

# =========================
# Test (체크포인트들 평가)
# =========================
test_ds = TFRecordKITTIDataset(config.TEST_TFRECORD_PATTERN, transform=pil_to_tensor, use_multi=True)

def evaluate_ckpt_on_test(model, ckpt_path):
    # 매 평가마다 새로운 DataLoader 생성 (IterableDataset 재사용 이슈 방지)
    test_loader = DataLoader(test_ds, batch_size=config.BATCH_SIZE, collate_fn=collate_fn, num_workers=0)
    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()
    total = 0.0
    n_batches = 0
    with torch.no_grad(), amp.autocast('cuda', enabled=torch.cuda.is_available()):
        for imgs, gt_boxes, cls_ids in test_loader:
            imgs = imgs.to(device)
            imgs, (cls_t, box_t) = label_encoder.encode_batch(imgs, gt_boxes, cls_ids)
            cls_t, box_t = cls_t.to(device), box_t.to(device)
            pred = model(imgs)
            loss = torch.mean(loss_fn(pred, (cls_t, box_t)))
            total += float(loss.detach().item())
            n_batches += 1
    avg = total / max(1, n_batches)
    print(f"{os.path.basename(ckpt_path)} | test loss {avg:.4f}")

ckpt_paths = sorted(
    glob.glob(config.EPOCH_CHECKPOINT_TEMPLATE.format(epoch='*')),
    key=lambda x: int(re.search(r'checkpoint_epoch_(\d+).pth', x).group(1))
)

if not ckpt_paths:
    print("No epoch checkpoints found.")
else:
    for p in ckpt_paths:
        evaluate_ckpt_on_test(model, p)


Python : 3.12.11 | packaged by conda-forge | (main, Jun  4 2025, 14:45:31) [GCC 13.3.0]
PyTorch: 2.7.1+cu118
CUDA   : 11.8 | is_available: True
[Info] use_multi=True가 전달되었지만, read-only/캐시 인덱스 호환을 위해 파일별 TFRecordDataset 체인 방식으로 대체합니다.
[Info] use_multi=True가 전달되었지만, read-only/캐시 인덱스 호환을 위해 파일별 TFRecordDataset 체인 방식으로 대체합니다.
[Resume] start from epoch 3
Epoch 4/10 | Step 10 | loss 0.2966
Epoch 4/10 | Step 20 | loss 0.2966
Epoch 4/10 | Step 30 | loss 0.2966
Epoch 4/10 | Step 40 | loss 0.2966
Epoch 4/10 | Step 50 | loss 0.2932
Epoch 4/10 | Step 60 | loss 0.2931
Epoch 4/10 | Step 70 | loss 0.2931
Epoch 4/10 | Step 80 | loss 0.2931
Epoch 4/10 | Step 90 | loss 0.2931
Epoch 4/10 | Step 100 | loss 0.2931
Epoch 4/10 | Step 110 | loss 0.2931
Epoch 4/10 | Step 120 | loss 0.2931
Epoch 4/10 | Step 130 | loss 0.2931
Epoch 4/10 | Step 140 | loss 0.2931
Epoch 4/10 | Step 150 | loss 0.2931
Epoch 4/10 | Step 160 | loss 0.2931
Epoch 4/10 | Step 170 | loss 0.2931
Epoch 4/10 | Step 180 | loss 0.2931
Epoch 4/1

KeyboardInterrupt: 

In [16]:

# =========================
# STOP/GO by two rules only (Person or Big Vehicle)
# =========================

# KITTI 8클래스 가정: 0:Car, 1:Van, 2:Truck, 3:Pedestrian, 4:Person_sitting, 5:Cyclist, 6:Tram, 7:Misc
PERSON_CLASS_IDS  = {3, 4}          # 사람
VEHICLE_CLASS_IDS = {0, 1, 2, 6}    # 차량 계열
SCORE_TH = 0.30
BIG_PIX  = 300

decoder = DecodePredictions().to(device)
decoder.eval()

def _boxes_to_original_scale(boxes_resized, orig_w, orig_h, img_size=IMG_SIZE):
    """리사이즈(IMG_SIZE) 기준 박스를 원본 해상도로 환산"""
    if boxes_resized.numel() == 0:
        return boxes_resized
    boxes = boxes_resized.clone()
    sx = float(orig_w) / float(img_size)
    sy = float(orig_h) / float(img_size)
    boxes[:, [0, 2]] = boxes[:, [0, 2]] * sx
    boxes[:, [1, 3]] = boxes[:, [1, 3]] * sy
    return boxes

def _maybe_scale_gt_boxes_to_original(bboxes, orig_w, orig_h):
    """
    GT bbox가 [0~1] 정규화로 들어오는 경우를 대비해 원본 픽셀로 스케일.
    (최댓값이 1.5 이하이면 정규화로 간주)
    """
    if bboxes.numel() == 0:
        return bboxes
    if float(bboxes.max()) <= 1.5:
        scale = torch.tensor([orig_w, orig_h, orig_w, orig_h],
                             dtype=torch.float32, device=bboxes.device)  # ✅ 같은 디바이스
        return bboxes * scale
    return bboxes

def predict_image_boxes_two_rules(model, image_path):
    """
    이미지 1장에 대해 모델로 탐지 → 원본 해상도 좌표로 환산 → 두 가지 규칙만으로 Stop/Go 판단
    """
    img_pil = Image.open(image_path).convert('RGB')
    orig_w, orig_h = img_pil.size

    x = pil_to_tensor(img_pil).unsqueeze(0).to(device)
    model.eval()
    with torch.no_grad(), amp.autocast('cuda', enabled=torch.cuda.is_available()):
        cls_logits, box_preds = model(x)
        boxes_resized, scores, classes = decoder(x, (cls_logits, box_preds))

    # 원본 크기로 환산
    boxes = _boxes_to_original_scale(boxes_resized, orig_w, orig_h, IMG_SIZE)

    # --- 두 가지 규칙만 ---
    if boxes.numel() > 0:
        person_ids  = torch.tensor(list(PERSON_CLASS_IDS),  device=classes.device, dtype=classes.dtype)
        vehicle_ids = torch.tensor(list(VEHICLE_CLASS_IDS), device=classes.device, dtype=classes.dtype)

        # 1) 사람이 한 명 이상(점수 조건 포함)
        person_mask = torch.isin(classes, person_ids) & (scores > SCORE_TH)
        if person_mask.any():
            return "Stop"

        # 2) 큰 차량(폭 또는 높이 >= 300px)
        vehicle_mask = torch.isin(classes, vehicle_ids) & (scores > SCORE_TH)
        if vehicle_mask.any():
            v = boxes[vehicle_mask]
            w = (v[:,2] - v[:,0])
            h = (v[:,3] - v[:,1])
            if ((w >= BIG_PIX) | (h >= BIG_PIX)).any():
                return "Stop"

    return "Go"

# =========================
# Build evaluation set from test TFRecord (GT-based labels)
# =========================
def build_eval_list_from_test_tfrecord(limit=None):
    """
    test TFRecord에서 (이미지경로, GT라벨[Stop/Go]) 리스트를 구성.
    - TFRecord의 이미지를 임시 폴더에 저장하고 경로를 반환.
    - GT라벨은 '사람 존재 or 차량 bbox>=300px' 두 조건만으로 생성.
    """
    save_dir = os.path.join(config.ROOT_DIR, "data", "eval_cache")
    os.makedirs(save_dir, exist_ok=True)

    eval_list = []
    count = 0

    # 변형 없이 원본 PIL에서 크기 얻기 위해 transform=None 데이터셋을 별도로 사용
    raw_ds = TFRecordKITTIDataset(config.TEST_TFRECORD_PATTERN, transform=None, use_multi=True)
    for sample in raw_ds:
        img_pil = sample['image']              # PIL.Image (원본 크기)
        orig_w, orig_h = img_pil.size

        b = sample['bbox']                     # (xmin,ymin,xmax,ymax) float
        c = sample['class_id']                 # int64

        # GT bbox를 원본 픽셀 기준으로 맞춤
        b = _maybe_scale_gt_boxes_to_original(b, orig_w, orig_h)

        # --- 두 가지 규칙으로 GT Stop/Go 생성 ---
        gt_stop = False
        if b.numel() > 0:
            if torch.isin(c, torch.tensor(list(PERSON_CLASS_IDS))):
                gt_stop = True
            else:
                vehicle_mask = torch.isin(c, torch.tensor(list(VEHICLE_CLASS_IDS)))
                if vehicle_mask.any():
                    vb = b[vehicle_mask]
                    vw = (vb[:,2] - vb[:,0])
                    vh = (vb[:,3] - vb[:,1])
                    if ((vw >= BIG_PIX) | (vh >= BIG_PIX)).any():
                        gt_stop = True

        gt_label = "Stop" if gt_stop else "Go"

        # 이미지 파일로 저장 (파일명 충돌 방지용 카운터/원래 이름 혼합)
        fname = sample.get('filename', f"test_{count}.png")
        base = os.path.basename(str(fname))
        out_path = os.path.join(save_dir, f"{count:06d}__{base}")
        img_pil.save(out_path)

        eval_list.append((out_path, gt_label))
        count += 1
        if (limit is not None) and (count >= limit):
            break

    return eval_list

# =========================
# Evaluate on many test samples created from TFRecord GT
# =========================
def evaluate_two_rule_system(model, limit=None, verbose_every=50):
    """
    limit: 평가 샘플 수 제한(None이면 전체)
    """
    eval_data = build_eval_list_from_test_tfrecord(limit=limit)
    n = len(eval_data)
    if n == 0:
        print("No test samples found.")
        return

    correct = 0
    for i, (img_path, gt) in enumerate(eval_data, 1):
        pred = predict_image_boxes_two_rules(model, img_path)
        if pred == gt:
            correct += 1
        if verbose_every and (i % verbose_every == 0):
            print(f"[{i:5d}/{n}] GT:{gt} | Pred:{pred}")

    acc = correct / n
    score = int(round(acc * 100))
    print("\n=== Evaluation (two-rule Stop/Go) ===")
    print(f"Samples: {n}  |  Correct: {correct}  |  Accuracy: {acc*100:.2f}%  |  Score: {score}/100")
    return score, acc

# =========================
# Quick run examples
# =========================
# 1) 단일 이미지 판정 예시
img_path = os.path.join(config.ROOT_DIR, "data", "stop_3.png")
print("Decision:", predict_image_boxes_two_rules(model, img_path))

# 2) 대량 평가 (예: 500장만 빠르게)
evaluate_two_rule_system(model, limit=500, verbose_every=100)

# 3) 전체 test 세트 평가
# evaluate_two_rule_system(model, limit=None, verbose_every=200)

Decision: Go
[Info] use_multi=True가 전달되었지만, read-only/캐시 인덱스 호환을 위해 파일별 TFRecordDataset 체인 방식으로 대체합니다.


RuntimeError: Boolean value of Tensor with more than one value is ambiguous

In [22]:
# !pip install -q tfrecord pillow matplotlib torchvision --upgrade

import sys, os, torch
print("Python :", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA   :", torch.version.cuda, "| is_available:", torch.cuda.is_available())

import os, io, re, glob, time, math, itertools, copy, hashlib
from typing import Dict, Any, List, Tuple

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, IterableDataset
from torchvision import transforms, models
from torchvision.ops import nms

# =========================
# Config
# =========================
class TrainingConfig:
    ROOT_DIR = os.path.join(os.getenv("HOME") or "", "work/object_detection")

    # Data
    DATA_PATH = os.path.join(ROOT_DIR, "data")
    TRAIN_TFRECORD_PATTERN      = os.path.join(DATA_PATH, "kitti/3.2.0/kitti-train.tfrecord-*")
    VALIDATION_TFRECORD_PATTERN = os.path.join(DATA_PATH, "kitti/3.2.0/kitti-validation.tfrecord-*")
    TEST_TFRECORD_PATTERN       = os.path.join(DATA_PATH, "kitti/3.2.0/kitti-test.tfrecord-*")

    # Model
    NUM_CLASSES = 8

    # Training
    BATCH_SIZE = 16
    NUM_EPOCHS = 10
    LEARNING_RATES = [0.025, 0.0025, 0.0025, 0.0025, 0.0025, 0.0025]
    LEARNING_RATE_BOUNDARIES = [125, 250, 500, 240000, 360000]
    OPTIMIZER_MOMENTUM = 0.9

    # Loss parameters
    CLASSIFICATION_LOSS_ALPHA = 0.25
    CLASSIFICATION_LOSS_GAMMA = 2.0
    BOX_LOSS_DELTA = 1.0

    # Decoder parameters
    CONFIDENCE_THRESHOLD = 0.05
    NMS_IOU_THRESHOLD = 0.5
    MAX_DETECTIONS_PER_CLASS = 100
    MAX_DETECTIONS = 100
    BOX_VARIANCE = [0.1, 0.1, 0.2, 0.2]

    # Checkpoint
    CHECKPOINT_PATH = os.path.join(ROOT_DIR, 'checkpoints')
    LATEST_CHECKPOINT_FILE = os.path.join(CHECKPOINT_PATH, 'latest_checkpoint.pth')
    BEST_CHECKPOINT_FILE   = os.path.join(CHECKPOINT_PATH, 'best_checkpoint.pth')
    EPOCH_CHECKPOINT_TEMPLATE = os.path.join(CHECKPOINT_PATH, 'checkpoint_epoch_{epoch}.pth')

config = TrainingConfig()
os.makedirs(config.CHECKPOINT_PATH, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

# =========================
# TFRecord helpers
# =========================
try:
    from tfrecord.torch.dataset import TFRecordDataset, MultiTFRecordDataset
except ModuleNotFoundError:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "tfrecord"])
    from tfrecord.torch.dataset import TFRecordDataset, MultiTFRecordDataset

def _index_cache_dir() -> str:
    root = os.path.join(os.getenv("HOME") or "", "work/object_detection", "index_cache")
    os.makedirs(root, exist_ok=True)
    return root

def _hashed_index_path(tfrecord_path: str) -> str:
    h = hashlib.sha1(tfrecord_path.encode("utf-8")).hexdigest()[:16]
    base = os.path.basename(tfrecord_path)
    return os.path.join(_index_cache_dir(), f"{base}.{h}.index")

def ensure_index_file(data_path: str) -> str:
    """
    캐시에 index가 없거나 비정상(아주 작음)이면 재생성.
    임시 파일로 만든 뒤 os.replace로 원자 교체.
    """
    from tfrecord.tools.tfrecord2idx import create_index
    cache_idx = _hashed_index_path(data_path)

    def _bad(p: str) -> bool:
        return (not os.path.exists(p)) or (os.path.getsize(p) < 16)

    if _bad(cache_idx):
        tmp = cache_idx + ".tmp"
        if os.path.exists(tmp):
            try: os.remove(tmp)
            except OSError: pass
        create_index(data_path, tmp)
        os.replace(tmp, cache_idx)
    return cache_idx

# =========================
# Dataset / Collate
# =========================
class TFRecordKITTIDataset(IterableDataset):
    def __init__(self, tfrecord_pattern: str, transform=None, use_multi=False):
        super().__init__()
        self.user_transform = transform
        self.file_paths = sorted([p for p in glob.glob(tfrecord_pattern) if not p.endswith('.index')])
        if not self.file_paths:
            raise FileNotFoundError(f"No TFRecord files found for pattern: {tfrecord_pattern}")

        self.feature_description = {
            'image': "byte", 'image/file_name': "byte",
            'objects/bbox': "float", 'objects/type': "int",
        }

        if use_multi:
            print("[Info] use_multi=True가 전달되었지만, read-only/캐시 인덱스 호환을 위해 "
                  "파일별 TFRecordDataset 체인 방식으로 대체합니다.")

        self.datasets = []
        for p in self.file_paths:
            idx = ensure_index_file(p)
            ds = TFRecordDataset(
                data_path=p, index_path=idx,
                description=self.feature_description,
                transform=self._parse_record,
                shuffle_queue_size=1024
            )
            self.datasets.append(ds)

    def _parse_record(self, features: Dict[str, Any]) -> Dict[str, Any]:
        image = Image.open(io.BytesIO(features['image'])).convert('RGB')
        fname_raw = features.get('image/file_name', b'')
        filename = fname_raw.decode('utf-8', errors='ignore') if isinstance(fname_raw, bytes) else str(fname_raw)

        b_raw = features.get('objects/bbox')
        if b_raw is not None and len(b_raw) > 0:
            b = torch.tensor(b_raw, dtype=torch.float32).view(-1, 4)
            b = b[:, [1, 0, 3, 2]]  # (ymin,xmin,ymax,xmax)->(xmin,ymin,xmax,ymax)
        else:
            b = torch.empty((0, 4), dtype=torch.float32)

        t_raw = features.get('objects/type')
        labels = torch.tensor(t_raw, dtype=torch.int64) if (t_raw is not None and len(t_raw) > 0) \
                 else torch.empty((0,), dtype=torch.int64)

        sample = {'image': image, 'bbox': b, 'class_id': labels, 'filename': filename}
        if self.user_transform is not None:
            sample['image'] = self.user_transform(sample['image'])
        return sample

    def __iter__(self):
        return itertools.chain.from_iterable(self.datasets)

def collate_fn(batch: List[Dict[str, Any]]) -> Tuple[torch.Tensor, List[torch.Tensor], List[torch.Tensor]]:
    images = [item['image'] for item in batch]
    bboxes = [item['bbox'] for item in batch]
    class_ids = [item['class_id'] for item in batch]

    for i, img in enumerate(images):
        if not torch.is_tensor(img) or img.dim() != 3:
            raise TypeError(f"Image at {i} is not Tensor[C,H,W]: {type(img)}, shape={getattr(img,'shape',None)}")

    C = images[0].shape[0]
    max_h = max(int(img.shape[1]) for img in images)
    max_w = max(int(img.shape[2]) for img in images)

    padded = torch.zeros(len(images), C, max_h, max_w, dtype=images[0].dtype)
    for i, img in enumerate(images):
        _, h, w = img.shape
        padded[i, :, :h, :w] = img
    return padded, bboxes, class_ids

# =========================
# Geometry / Anchors
# =========================
def convert_to_xywh(boxes):
    return torch.cat([(boxes[...,:2] + boxes[...,2:]) / 2.0,
                      boxes[...,2:] - boxes[...,:2]], dim=-1)

def convert_to_corners(boxes):
    return torch.cat([boxes[...,:2] - boxes[...,2:]/2.0,
                      boxes[...,:2] + boxes[...,2:]/2.0], dim=-1)

def compute_iou(boxes1, boxes2):
    b1 = convert_to_corners(boxes1)
    b2 = convert_to_corners(boxes2)
    lu = torch.maximum(b1[:,None,:2], b2[:,:2])
    rd = torch.minimum(b1[:,None,2:], b2[:,2:])
    inter = torch.clamp(rd - lu, min=0.0)
    inter_area = inter[:,:,0] * inter[:,:,1]
    a1 = boxes1[:,2] * boxes1[:,3]
    a2 = boxes2[:,2] * boxes2[:,3]
    union = torch.clamp(a1[:,None] + a2 - inter_area, min=1e-8)
    return torch.clamp(inter_area / union, 0.0, 1.0)

class AnchorBox:
    def __init__(self):
        self.aspect_ratios = [0.5, 1.0, 2.0]
        self.scales = [2 ** x for x in [0, 1/3, 2/3]]
        self._num_anchors = len(self.aspect_ratios) * len(self.scales)
        self._strides = [2 ** i for i in range(3, 8)]
        self._areas = [x**2 for x in [32.0, 64.0, 128.0, 256.0, 512.0]]
        self._anchor_dims = self._compute_dims()

    def _compute_dims(self):
        out = []
        for area in self._areas:
            dims = []
            for r in self.aspect_ratios:
                h = torch.sqrt(torch.tensor(area / r))
                w = area / h
                for s in self.scales:
                    dims.append(torch.tensor([w*s, h*s]))
            out.append(torch.stack(dims, dim=0).view(1,1,self._num_anchors,2))
        return out

    def _get_anchors(self, fh, fw, level):
        rx = torch.arange(fw, dtype=torch.float32) + 0.5
        ry = torch.arange(fh, dtype=torch.float32) + 0.5
        centers = torch.stack(torch.meshgrid(rx, ry, indexing='xy'), dim=-1) * self._strides[level-3]
        centers = centers.unsqueeze(-2).repeat(1,1,self._num_anchors,1)
        dims = self._anchor_dims[level-3].repeat(fh, fw, 1, 1)
        anchors = torch.cat([centers, dims], dim=-1)
        return anchors.view(fh*fw*self._num_anchors, 4)

    def get_anchors(self, H, W):
        anchors = []
        for i in range(3,8):
            anchors.append(self._get_anchors(math.ceil(H/2**i), math.ceil(W/2**i), i))
        return torch.cat(anchors, dim=0)

# =========================
# Label Encoder
# =========================
class LabelEncoder:
    def __init__(self):
        self._anchor_box = AnchorBox()
        self._box_variance = torch.tensor(TrainingConfig.BOX_VARIANCE, dtype=torch.float32)

    def _match_anchor_boxes(self, anchors, gt_boxes, match_iou=0.5, ignore_iou=0.4):
        iou_matrix = compute_iou(anchors, gt_boxes)
        max_iou, matched_idx = torch.max(iou_matrix, dim=1)
        positive = max_iou >= match_iou
        negative = max_iou < ignore_iou
        ignore = ~(positive | negative)
        return matched_idx, positive.float(), ignore.float()

    def _compute_box_target(self, anchors, matched_gt):
        box_t = torch.cat([
            (matched_gt[:,:2] - anchors[:,:2]) / anchors[:,2:],
            torch.log(matched_gt[:,2:] / anchors[:,2:])
        ], dim=-1)
        return box_t / self._box_variance

    def _encode_single(self, image_shape, gt_boxes, cls_ids):
        anchors = self._anchor_box.get_anchors(image_shape[2], image_shape[3])
        if gt_boxes.numel() == 0:
            box_target = torch.zeros_like(anchors)
            cls_target = torch.full((anchors.shape[0],), -1.0, dtype=torch.float32)
            return box_target, cls_target

        gt_xywh = convert_to_xywh(gt_boxes)
        matched_idx, positive, ignore = self._match_anchor_boxes(anchors, gt_xywh)
        matched_gt = gt_xywh[matched_idx]
        box_target = self._compute_box_target(anchors, matched_gt)

        cls_ids = cls_ids.float()
        matched_cls = cls_ids[matched_idx]
        cls_target = torch.where(positive!=1.0, -1.0, matched_cls)
        cls_target = torch.where(ignore==1.0, -2.0, cls_target)
        return box_target, cls_target

    def encode_batch(self, images, gt_boxes_list, cls_ids_list):
        box_targets = []
        cls_targets = []
        for i in range(images.size(0)):
            b, c = self._encode_single(images.size(), gt_boxes_list[i], cls_ids_list[i])
            box_targets.append(b); cls_targets.append(c)
        box_targets = torch.stack(box_targets, dim=0)
        cls_targets = torch.stack(cls_targets, dim=0)
        return images, (cls_targets, box_targets)

# =========================
# Model
# =========================
class ResNetBackbone(nn.Module):
    def __init__(self):
        super().__init__()
        resnet = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        self.layer1 = nn.Sequential(*list(resnet.children())[:5])  # stem + layer1
        self.layer2 = resnet.layer2
        self.layer3 = resnet.layer3
        self.layer4 = resnet.layer4

    def forward(self, x):
        c1 = self.layer1(x)
        c2 = self.layer2(c1)
        c3 = self.layer3(c2)
        c4 = self.layer4(c3)
        return c2, c3, c4

class FeaturePyramid(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone
        self.conv_c3_1x1 = nn.Conv2d( 512, 256, 1)
        self.conv_c4_1x1 = nn.Conv2d(1024, 256, 1)
        self.conv_c5_1x1 = nn.Conv2d(2048, 256, 1)
        self.conv_c3_3x3 = nn.Conv2d(256, 256, 3, padding=1)
        self.conv_c4_3x3 = nn.Conv2d(256, 256, 3, padding=1)
        self.conv_c5_3x3 = nn.Conv2d(256, 256, 3, padding=1)
        self.conv_c6_3x3 = nn.Conv2d(2048, 256, 3, stride=2, padding=1)
        self.conv_c7_3x3 = nn.Conv2d(256, 256, 3, stride=2, padding=1)

    def forward(self, x):
        c3, c4, c5 = self.backbone(x)
        p5 = self.conv_c5_1x1(c5)
        p4_lat = self.conv_c4_1x1(c4)
        p4  = p4_lat + F.interpolate(p5, size=p4_lat.shape[2:], mode='nearest')
        p3_lat = self.conv_c3_1x1(c3)
        p3  = p3_lat + F.interpolate(p4, size=p3_lat.shape[2:], mode='nearest')
        p3 = self.conv_c3_3x3(p3); p4 = self.conv_c4_3x3(p4); p5 = self.conv_c5_3x3(p5)
        p6 = self.conv_c6_3x3(c5)
        p7 = self.conv_c7_3x3(F.relu(p6))
        return p3, p4, p5, p6, p7

def build_head(out_filters, bias_init):
    layers = []
    for _ in range(4):
        layers += [nn.Conv2d(256,256,3,padding=1), nn.ReLU()]
    last = nn.Conv2d(256, out_filters, 3, padding=1)
    if isinstance(bias_init, float):
        nn.init.constant_(last.bias, bias_init)
    else:
        nn.init.constant_(last.bias, 0.0)
    layers.append(last)
    return nn.Sequential(*layers)

class RetinaNet(nn.Module):
    def __init__(self, num_classes, backbone=None):
        super().__init__()
        self.num_classes = num_classes
        self.fpn = FeaturePyramid(backbone or ResNetBackbone())
        prior_probability = -torch.log(torch.tensor((1 - 0.01) / 0.01))
        self.cls_head = build_head(9*num_classes, float(prior_probability))
        self.box_head = build_head(9*4, "zeros")

    def forward(self, x):
        feats = self.fpn(x)
        N = x.size(0)
        cls_outs, box_outs = [], []
        for f in feats:
            box_outs.append(self.box_head(f).view(N, -1, 4))
            cls_outs.append(self.cls_head(f).view(N, -1, self.num_classes))
        return torch.cat(cls_outs, 1), torch.cat(box_outs, 1)

# =========================
# Decode & Viz
# =========================
class DecodePredictions(nn.Module):
    def __init__(self):
        super().__init__()
        self.num_classes = TrainingConfig.NUM_CLASSES
        self.confidence_threshold = TrainingConfig.CONFIDENCE_THRESHOLD
        self.nms_iou_threshold = TrainingConfig.NMS_IOU_THRESHOLD
        self.max_detections_per_class = TrainingConfig.MAX_DETECTIONS_PER_CLASS
        self.max_detections = TrainingConfig.MAX_DETECTIONS
        self._anchors = AnchorBox()
        self.register_buffer("_box_variance",
                             torch.tensor(TrainingConfig.BOX_VARIANCE, dtype=torch.float32))

    def _decode(self, anchors, box_pred):
        boxes = box_pred * self._box_variance
        boxes = torch.cat([
            boxes[:, :, :2] * anchors[:, :, 2:] + anchors[:, :, :2],
            torch.exp(boxes[:, :, 2:]) * anchors[:, :, 2:]
        ], dim=-1)
        return convert_to_corners(boxes)

    def forward(self, images, preds):
        cls_logits, box_preds = preds
        cls_scores = torch.sigmoid(cls_logits)
        H, W = images.shape[2], images.shape[3]
        anchors = self._anchors.get_anchors(H, W)[None, ...].to(box_preds.device)
        boxes = self._decode(anchors, box_preds)

        out_boxes, out_scores, out_classes = [], [], []
        for b in range(images.size(0)):
            bboxes = boxes[b]
            scores = cls_scores[b]
            for c in range(self.num_classes):
                cs = scores[:, c]
                mask = cs > self.confidence_threshold
                if not mask.any():
                    continue
                c_boxes = bboxes[mask]; c_scores = cs[mask]
                if c_scores.numel() > self.max_detections_per_class:
                    topk = torch.topk(c_scores, self.max_detections_per_class)
                    c_scores = topk.values; c_boxes = c_boxes[topk.indices]
                keep = nms(c_boxes, c_scores, self.nms_iou_threshold)
                out_boxes.append(c_boxes[keep])
                out_scores.append(c_scores[keep])
                out_classes.append(torch.full_like(c_scores[keep], c, dtype=torch.int64))
        if not out_boxes:
            dev = box_preds.device
            return (torch.empty(0,4, device=dev),
                    torch.empty(0, device=dev),
                    torch.empty(0, dtype=torch.int64, device=dev))
        fb = torch.cat(out_boxes,0); fs = torch.cat(out_scores,0); fc = torch.cat(out_classes,0)
        if fs.numel() > self.max_detections:
            topk = torch.topk(fs, self.max_detections)
            fb, fs, fc = fb[topk.indices], topk.values, fc[topk.indices]
        return fb, fs, fc

def visualize_detections(image, boxes, classes, scores, figsize=(7,7), linewidth=1):
    img = np.array(image, dtype=np.uint8)
    plt.figure(figsize=figsize); plt.axis("off"); plt.imshow(img)
    ax = plt.gca()
    for box, c, s in zip(boxes, classes, scores):
        x1,y1,x2,y2 = box
        w,h = x2-x1, y2-y1
        rect = plt.Rectangle((x1,y1), w,h, fill=False, linewidth=linewidth)
        ax.add_patch(rect)
        ax.text(x1, y1, f"{int(c)}:{s:.2f}", bbox={"facecolor":"white","alpha":0.5}, fontsize=8)
    plt.show()

# =========================
# Loss
# =========================
class RetinaNetBoxLoss(nn.Module):
    def __init__(self, delta):
        super().__init__()
        self.delta = delta
    def forward(self, y_true, y_pred):
        d = y_true - y_pred
        ad = torch.abs(d)
        loss = torch.where(ad < self.delta, 0.5 * (d**2), ad - 0.5)
        return torch.sum(loss, dim=-1)

class RetinaNetClassificationLoss(nn.Module):
    def __init__(self, alpha, gamma):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
    def forward(self, y_true, y_pred):
        ce = F.binary_cross_entropy_with_logits(y_pred, y_true, reduction='none')
        p  = torch.sigmoid(y_pred)
        alpha = torch.where(y_true == 1.0, self.alpha, 1.0 - self.alpha)
        pt = torch.where(y_true == 1.0, p, 1.0 - p)
        loss = alpha * (1.0 - pt) ** self.gamma * ce
        return torch.sum(loss, dim=-1)

class RetinaNetLoss(nn.Module):
    def __init__(self, num_classes=TrainingConfig.NUM_CLASSES):
        super().__init__()
        self.num_classes = num_classes
        self.clf = RetinaNetClassificationLoss(TrainingConfig.CLASSIFICATION_LOSS_ALPHA,
                                               TrainingConfig.CLASSIFICATION_LOSS_GAMMA)
        self.box = RetinaNetBoxLoss(TrainingConfig.BOX_LOSS_DELTA)
    def forward(self, preds, targets):
        cls_logits, box_preds = preds
        cls_targets, box_targets = targets
        positive_mask = (cls_targets > -1.0).float()
        ignore_mask   = (cls_targets == -2.0).float()
        normalizer = torch.sum(positive_mask, dim=-1).clamp(min=1.0)

        temp = torch.where(cls_targets >= 0, cls_targets.long(), 0)
        one_hot = F.one_hot(temp, num_classes=self.num_classes).float()
        one_hot = one_hot * positive_mask.unsqueeze(-1)

        clf_loss = self.clf(one_hot, cls_logits.float())
        clf_loss = torch.where(ignore_mask==1.0, 0.0, clf_loss)
        clf_loss = torch.sum(clf_loss, dim=-1) / normalizer

        box_loss = self.box(box_targets.float(), box_preds.float())
        box_loss = torch.where(positive_mask==1.0, box_loss, 0.0)
        box_loss = torch.sum(box_loss, dim=-1) / normalizer

        return clf_loss + box_loss

# =========================
# Transform
# =========================
IMG_SIZE = 512

pil_to_tensor = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406],
                         std =[0.229,0.224,0.225]),
])

# =========================
# DataLoader
# =========================
train_ds = TFRecordKITTIDataset(config.TRAIN_TFRECORD_PATTERN, transform=pil_to_tensor, use_multi=True)
val_ds   = TFRecordKITTIDataset(config.VALIDATION_TFRECORD_PATTERN, transform=pil_to_tensor, use_multi=True)

train_loader = DataLoader(train_ds, batch_size=config.BATCH_SIZE, collate_fn=collate_fn, num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=config.BATCH_SIZE, collate_fn=collate_fn, num_workers=0)

# =========================
# Model / Optim / Sched
# =========================
model = RetinaNet(config.NUM_CLASSES).to(device)
label_encoder = LabelEncoder()
loss_fn = RetinaNetLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=config.LEARNING_RATES[0], momentum=config.OPTIMIZER_MOMENTUM)

def lr_lambda(epoch):
    for i,b in enumerate(config.LEARNING_RATE_BOUNDARIES):
        if epoch < b:
            return config.LEARNING_RATES[i] / config.LEARNING_RATES[0]
    return config.LEARNING_RATES[-1] / config.LEARNING_RATES[0]

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

# =========================
# Train / Eval (AMP 최신 API)
# =========================
from torch import amp
scaler = amp.GradScaler('cuda', enabled=torch.cuda.is_available())

start_epoch, best_val = 0, float('inf')

if os.path.exists(config.LATEST_CHECKPOINT_FILE):
    ckpt = torch.load(config.LATEST_CHECKPOINT_FILE, map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    start_epoch = ckpt['epoch'] + 1
    print(f"[Resume] start from epoch {start_epoch}")

for epoch in range(start_epoch, config.NUM_EPOCHS):
    t0 = time.time()
    model.train()
    tr_loss = 0.0
    tr_batches = 0

    for i, (imgs, gt_boxes, cls_ids) in enumerate(train_loader):
        imgs = imgs.to(device, non_blocking=True)

        imgs, (cls_t, box_t) = label_encoder.encode_batch(imgs, gt_boxes, cls_ids)
        cls_t, box_t = cls_t.to(device, non_blocking=True), box_t.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with amp.autocast('cuda', enabled=torch.cuda.is_available()):
            pred = model(imgs)
            loss = torch.mean(loss_fn(pred, (cls_t, box_t)))

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        tr_loss += float(loss.detach().item())
        tr_batches += 1

        if (i+1) % 10 == 0:
            print(f"Epoch {epoch+1}/{config.NUM_EPOCHS} | Step {i+1} | loss {loss.item():.4f}")

        del imgs, cls_t, box_t, pred, loss
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    tr_loss /= max(1, tr_batches)

    # ===== Validation =====
    model.eval()
    va_loss = 0.0
    va_batches = 0
    with torch.no_grad(), amp.autocast('cuda', enabled=torch.cuda.is_available()):
        for imgs, gt_boxes, cls_ids in val_loader:
            imgs = imgs.to(device, non_blocking=True)
            imgs, (cls_t, box_t) = label_encoder.encode_batch(imgs, gt_boxes, cls_ids)
            cls_t, box_t = cls_t.to(device, non_blocking=True), box_t.to(device, non_blocking=True)

            pred = model(imgs)
            loss = torch.mean(loss_fn(pred, (cls_t, box_t)))
            va_loss += float(loss.detach().item())
            va_batches += 1

            del imgs, cls_t, box_t, pred, loss
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    va_loss /= max(1, va_batches)
    dt = time.time() - t0
    print(f"[Epoch {epoch+1}] train {tr_loss:.4f} | val {va_loss:.4f} | {dt:.1f}s")

    scheduler.step()

    # latest
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict()
    }, config.LATEST_CHECKPOINT_FILE)

    # per-epoch
    torch.save({'epoch': epoch, 'model_state_dict': model.state_dict()},
               config.EPOCH_CHECKPOINT_TEMPLATE.format(epoch=epoch+1))

    if va_loss < best_val:
        best_val = va_loss
        torch.save({'epoch': epoch, 'model_state_dict': model.state_dict()}, config.BEST_CHECKPOINT_FILE)
        print(f"** New Best Saved (val {va_loss:.4f}) **")

# =========================
# Test (체크포인트들 평가)
# =========================
test_ds = TFRecordKITTIDataset(config.TEST_TFRECORD_PATTERN, transform=pil_to_tensor, use_multi=True)

def evaluate_ckpt_on_test(model, ckpt_path):
    test_loader = DataLoader(test_ds, batch_size=config.BATCH_SIZE, collate_fn=collate_fn, num_workers=0)
    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()
    total = 0.0
    n_batches = 0
    with torch.no_grad(), amp.autocast('cuda', enabled=torch.cuda.is_available()):
        for imgs, gt_boxes, cls_ids in test_loader:
            imgs = imgs.to(device)
            imgs, (cls_t, box_t) = label_encoder.encode_batch(imgs, gt_boxes, cls_ids)
            cls_t, box_t = cls_t.to(device), box_t.to(device)
            pred = model(imgs)
            loss = torch.mean(loss_fn(pred, (cls_t, box_t)))
            total += float(loss.detach().item())
            n_batches += 1
    avg = total / max(1, n_batches)
    print(f"{os.path.basename(ckpt_path)} | test loss {avg:.4f}")

ckpt_paths = sorted(
    glob.glob(config.EPOCH_CHECKPOINT_TEMPLATE.format(epoch='*')),
    key=lambda x: int(re.search(r'checkpoint_epoch_(\d+).pth', x).group(1))
)

if not ckpt_paths:
    print("No epoch checkpoints found.")
else:
    for p in ckpt_paths:
        evaluate_ckpt_on_test(model, p)



Python : 3.12.11 | packaged by conda-forge | (main, Jun  4 2025, 14:45:31) [GCC 13.3.0]
PyTorch: 2.7.1+cu118
CUDA   : 11.8 | is_available: True
[Info] use_multi=True가 전달되었지만, read-only/캐시 인덱스 호환을 위해 파일별 TFRecordDataset 체인 방식으로 대체합니다.
[Info] use_multi=True가 전달되었지만, read-only/캐시 인덱스 호환을 위해 파일별 TFRecordDataset 체인 방식으로 대체합니다.
[Resume] start from epoch 3
Epoch 4/10 | Step 10 | loss 0.2966
Epoch 4/10 | Step 20 | loss 0.2966
Epoch 4/10 | Step 30 | loss 0.2966
Epoch 4/10 | Step 40 | loss 0.2966
Epoch 4/10 | Step 50 | loss 0.2932
Epoch 4/10 | Step 60 | loss 0.2931
Epoch 4/10 | Step 70 | loss 0.2931
Epoch 4/10 | Step 80 | loss 0.2931
Epoch 4/10 | Step 90 | loss 0.2931
Epoch 4/10 | Step 100 | loss 0.2931
Epoch 4/10 | Step 110 | loss 0.2931
Epoch 4/10 | Step 120 | loss 0.2931
Epoch 4/10 | Step 130 | loss 0.2931
Epoch 4/10 | Step 140 | loss 0.2931
Epoch 4/10 | Step 150 | loss 0.2931
Epoch 4/10 | Step 160 | loss 0.2931
Epoch 4/10 | Step 170 | loss 0.2931
Epoch 4/10 | Step 180 | loss 0.2931
Epoch 4/1

KeyboardInterrupt: 

In [23]:
# =========================
# STOP/GO by two rules only (Person or Big Vehicle)
# =========================
PERSON_CLASS_IDS  = {3, 4}          # 사람
VEHICLE_CLASS_IDS = {0, 1, 2, 6}    # 차량 계열
SCORE_TH = 0.30
BIG_PIX  = 300

decoder = DecodePredictions().to(device)
decoder.eval()

def _boxes_to_original_scale(boxes_resized, orig_w, orig_h, img_size=IMG_SIZE):
    if boxes_resized.numel() == 0:
        return boxes_resized
    boxes = boxes_resized.clone()
    sx = float(orig_w) / float(img_size)
    sy = float(orig_h) / float(img_size)
    boxes[:, [0, 2]] = boxes[:, [0, 2]] * sx
    boxes[:, [1, 3]] = boxes[:, [1, 3]] * sy
    return boxes

def _maybe_scale_gt_boxes_to_original(bboxes, orig_w, orig_h):
    if bboxes.numel() == 0:
        return bboxes
    if float(bboxes.max()) <= 1.5:
        scale = torch.tensor([orig_w, orig_h, orig_w, orig_h],
                             dtype=torch.float32, device=bboxes.device)
        return bboxes * scale
    return bboxes

def predict_image_boxes_two_rules(model, image_path):
    img_pil = Image.open(image_path).convert('RGB')
    orig_w, orig_h = img_pil.size

    x = pil_to_tensor(img_pil).unsqueeze(0).to(device)
    model.eval()
    with torch.no_grad(), amp.autocast('cuda', enabled=torch.cuda.is_available()):
        cls_logits, box_preds = model(x)
        boxes_resized, scores, classes = decoder(x, (cls_logits, box_preds))

    boxes = _boxes_to_original_scale(boxes_resized, orig_w, orig_h, IMG_SIZE)

    if boxes.numel() > 0:
        person_ids  = torch.tensor(list(PERSON_CLASS_IDS),  device=classes.device, dtype=classes.dtype)
        vehicle_ids = torch.tensor(list(VEHICLE_CLASS_IDS), device=classes.device, dtype=classes.dtype)

        person_mask = torch.isin(classes, person_ids) & (scores > SCORE_TH)
        if person_mask.any().item():
            return "Stop"

        vehicle_mask = torch.isin(classes, vehicle_ids) & (scores > SCORE_TH)
        if vehicle_mask.any().item():
            v = boxes[vehicle_mask]
            w = (v[:,2] - v[:,0])
            h = (v[:,3] - v[:,1])
            if ((w >= BIG_PIX) | (h >= BIG_PIX)).any().item():
                return "Stop"

    return "Go"

# =========================
# Build evaluation set from test TFRecord (GT-based labels)
# =========================
def build_eval_list_from_test_tfrecord(limit=None):
    save_dir = os.path.join(config.ROOT_DIR, "data", "eval_cache")
    os.makedirs(save_dir, exist_ok=True)

    eval_list = []
    count = 0

    raw_ds = TFRecordKITTIDataset(config.TEST_TFRECORD_PATTERN, transform=None, use_multi=True)
    for sample in raw_ds:
        img_pil = sample['image']
        orig_w, orig_h = img_pil.size

        b = sample['bbox']          # (xmin,ymin,xmax,ymax)
        c = sample['class_id']      # shape [N], int64

        b = _maybe_scale_gt_boxes_to_original(b, orig_w, orig_h)

        # ====== 여기 수정: isin 결과는 벡터 → any().item()으로 스칼라 변환 ======
        gt_stop = False
        if b.numel() > 0:
            person_ids = torch.tensor(list(PERSON_CLASS_IDS), dtype=c.dtype, device=c.device)
            if torch.isin(c, person_ids).any().item():
                gt_stop = True
            else:
                vehicle_ids = torch.tensor(list(VEHICLE_CLASS_IDS), dtype=c.dtype, device=c.device)
                vehicle_mask = torch.isin(c, vehicle_ids)
                if vehicle_mask.any().item():
                    vb = b[vehicle_mask]
                    vw = (vb[:,2] - vb[:,0])
                    vh = (vb[:,3] - vb[:,1])
                    if ((vw >= BIG_PIX) | (vh >= BIG_PIX)).any().item():
                        gt_stop = True
        # ================================================================

        gt_label = "Stop" if gt_stop else "Go"

        fname = sample.get('filename', f"test_{count}.png")
        base = os.path.basename(str(fname))
        out_path = os.path.join(save_dir, f"{count:06d}__{base}")
        img_pil.save(out_path)

        eval_list.append((out_path, gt_label))
        count += 1
        if (limit is not None) and (count >= limit):
            break

    return eval_list

# =========================
# Evaluate on many test samples created from TFRecord GT
# =========================
def evaluate_two_rule_system(model, limit=None, verbose_every=50):
    eval_data = build_eval_list_from_test_tfrecord(limit=limit)
    n = len(eval_data)
    if n == 0:
        print("No test samples found.")
        return

    correct = 0
    for i, (img_path, gt) in enumerate(eval_data, 1):
        pred = predict_image_boxes_two_rules(model, img_path)
        if pred == gt:
            correct += 1
        if verbose_every and (i % verbose_every == 0):
            print(f"[{i:5d}/{n}] GT:{gt} | Pred:{pred}")

    acc = correct / n
    score = int(round(acc * 100))
    print("\n=== Evaluation (two-rule Stop/Go) ===")
    print(f"Samples: {n}  |  Correct: {correct}  |  Accuracy: {acc*100:.2f}%  |  Score: {score}/100")
    return score, acc

# =========================
# Quick run examples
# =========================
# 1) 단일 이미지 판정 예시
img_path = os.path.join(config.ROOT_DIR, "data", "stop_3.png")
print("Decision:", predict_image_boxes_two_rules(model, img_path))

# 2) 대량 평가 (예: 500장만 빠르게)
evaluate_two_rule_system(model, limit=100, verbose_every=10)

# 3) 전체 test 세트 평가
# evaluate_two_rule_system(model, limit=None, verbose_every=200)


Decision: Go
[Info] use_multi=True가 전달되었지만, read-only/캐시 인덱스 호환을 위해 파일별 TFRecordDataset 체인 방식으로 대체합니다.
[   10/100] GT:Stop | Pred:Go
[   20/100] GT:Go | Pred:Go
[   30/100] GT:Stop | Pred:Go
[   40/100] GT:Go | Pred:Go
[   50/100] GT:Go | Pred:Go
[   60/100] GT:Go | Pred:Go
[   70/100] GT:Go | Pred:Go
[   80/100] GT:Go | Pred:Go
[   90/100] GT:Stop | Pred:Go
[  100/100] GT:Stop | Pred:Go

=== Evaluation (two-rule Stop/Go) ===
Samples: 100  |  Correct: 65  |  Accuracy: 65.00%  |  Score: 65/100


(65, 0.65)